# 📗 과적합과 규제, 그리고 Lightning: 손글씨 숫자 분류의 과적합 줄이기

학습 데이터에는 잘 맞는데 처음 보는 데이터에서는 성능이 떨어지는 상태를 과적합(overfitting)이라고 합니다. 학습 그림이 적고 파라미터가 많은 모델에서 잘 생깁니다. 이 교안은 교안 01의 손글씨 숫자 CNN으로 과적합을 일으켜 진단하고, 규제(regularization, 과적합을 줄이려고 학습에 거는 제약. 정규화라고도 부르지만 교안 01의 배치 정규화(batch normalization)와는 다른 개념)와 조기 종료로 줄인 뒤, 같은 학습을 PyTorch Lightning으로 다시 씁니다.

<img src="images/70_lesson02_goals.png" width="1050" alt="제목은 '이번 교안에서 배울 것: 손글씨 숫자 분류의 과적합 줄이기'이고, 제목 아래 띠에 'MNIST 손글씨 숫자 학습 800장, 검증 200장, 테스트 2,000장'이 적혀 있습니다. 판 네 개가 2행 2열로 놓여 왼쪽 위 1, 오른쪽 위 2, 왼쪽 아래 3, 오른쪽 아래 4 순서로 화살표로 이어집니다. 1 '과적합 진단'(칩 '묶음 단위 분할과 학습 곡선') 판에서는 파란 '학습 손실'과 빨간 '검증 손실'이 같은 점에서 출발해, 학습 손실은 바닥 가까이 내려가고 검증 손실은 늘 그보다 위에서 가장 낮은 점(별)을 지나 다시 오릅니다. 2 '규제 세 가지'(칩 'Dropout, weight decay, 데이터 증강') 판에는 뉴런 두 개에 X가 그려진 신경망, 짧아지는 초록 막대 네 개, 옆으로 옮긴 숫자 7이 있습니다. 3 '조기 종료와 테스트 한 번'(칩 '검증 손실이 가장 낮던 파라미터') 판에는 검증 손실 곡선의 가장 낮은 점에 별이 있고, 별에서 시작하는 회색 띠 'patience 10' 끝의 점선에 '멈춤'이 있습니다. 4 'PyTorch Lightning'(칩 'Trainer, 기울기 자르기, 이어 학습') 판에는 'LightningModule'과 'Trainer' 상자, Trainer 아래 'EarlyStopping', 'ModelCheckpoint', 'CSVLogger' 상자가 있고, 개발자(초록 옷)가 Trainer를 가리킵니다.">


**학습 목표** (그림의 번호와 같습니다)

1. **데이터 누수가 없게 묶음 단위로 학습·검증·테스트 데이터를 나누는 법을 알고, 학습 손실과 검증 손실 곡선으로 과적합을 진단합니다.**
2. **Dropout·weight decay·데이터 증강을 CNN에 더하고, 검증 손실로 효과를 비교합니다.**
3. **조기 종료로 검증 손실이 가장 낮던 파라미터를 남기고, 검증 손실로 고른 설정을 테스트 데이터로 한 번 평가합니다.**
4. **같은 학습을 PyTorch Lightning의 `LightningModule`과 `Trainer`(EarlyStopping, ModelCheckpoint, CSVLogger)로 정리하고, 기울기 자르기와 체크포인트에서 이어 학습하기를 더합니다.**

## 1. 실습 준비

과적합을 보려면 학습에 쓰지 않은 데이터가 따로 있어야 하므로, 데이터를 셋으로 나누는 것부터 합니다.

### 1-1. 라이브러리와 데이터

준비 셀은 교안 01과 같습니다(계산 장치와 그래프 글꼴). 데이터는 교안 01과 같은 손글씨 숫자 [MNIST](https://web.archive.org/web/20231231235959/http://yann.lecun.com/exdb/mnist/)(LeCun, Cortes & Burges, CC BY-SA 3.0) 표본입니다(`data/본실습/mnist_small.pt`).

In [ ]:
import platform
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
# torch.load는 학습용과 테스트용 그림, 정답, 종류 이름이 든 딕셔너리를 돌려줍니다
mnist = torch.load(data_dir / "본실습" / "mnist_small.pt")
print("학습용:", mnist["train_images"].shape, "| 테스트용:", mnist["test_images"].shape)

### 1-2. 학습·검증·테스트 나누기

지난 시간에는 설정(학습률, 옵티마이저)을 학습 손실로 골랐습니다. 그런데 학습 손실이 낮은 설정이 처음 보는 데이터에서도 나은지는 학습 손실로 알 수 없으므로, 학습에 쓰지 않는 데이터를 하나 더 둡니다.

- **학습 데이터:** 파라미터를 갱신하는 데 씁니다.
- **검증 데이터(validation data):** 학습에 쓰지 않고, 설정(하이퍼파라미터. 규제 종류, 멈출 에폭)을 고를 때만 봅니다.
- **테스트 데이터:** 설정을 다 고른 뒤 마지막에 한 번만 봅니다. 검증 데이터는 고르는 데 썼으므로, 고른 모델의 성능은 고르는 데 쓰지 않은 테스트 데이터로 보고합니다.


<img src="images/71_data_split.png" width="1050" alt="제목은 '학습 · 검증 · 테스트 데이터의 역할'입니다. 위쪽 긴 띠가 '학습용 10,000장 가운데 1,000장'에서 '학습 800장'(파랑)과 '검증 200장'(빨강)으로 나뉘고, 오른쪽에 떨어진 '테스트 2,000장'(보라) 띠가 따로 있습니다. 띠마다 아래 화살표로 역할 상자가 이어집니다. 파랑 상자 '파라미터 갱신', 빨강 상자 '설정 고르기(규제 종류, 멈출 에폭)', 보라 상자 '마지막에 한 번 평가'. 맨 아래 띠에 '테스트 결과를 보고 설정을 다시 바꾸지 않습니다'가 적혀 있습니다. 개발자(초록 옷)가 빨강 상자 옆에서 두 그래프를 비교합니다.">


**데이터 누수(data leakage)**
- 검증·테스트 데이터의 정보가 학습에 새어 들어가, 검증·테스트 성능이 처음 보는 데이터에서의 실제 성능보다 좋게 나오는 것입니다. 누수를 막는 목적은 검증·테스트 결과가 처음 보는 데이터의 성능을 그대로 보여 주게 하는 것입니다.
- **같은 원본이 양쪽에 들어감:** 같은 사람이 쓰거나 찍은 데이터, 같은 그림의 사본, 한 그림을 증강해 만든 그림이 학습과 검증에 나뉘면, 모델은 검증 그림과 거의 같은 그림을 이미 학습한 상태로 평가받습니다.
- **학습 데이터 밖의 값을 학습에 씀:** 표준화의 평균과 표준편차는 학습 데이터로만 계산합니다. 딥러닝 학습 원리 단원 교안 02 따라하기에서 학습용 평균과 표준편차로 테스트용까지 표준화한 이유입니다.
- **테스트 결과로 설정을 고름:** 테스트 데이터가 고르는 데 쓴 데이터가 됩니다(7-2 확인 퀴즈).

**묶음 단위 분할(group split):** 같은 사람이나 같은 원본에서 나온 데이터를 한 묶음으로 보고, 묶음째 학습이나 검증 한쪽에만 넣는 분할입니다. 과제 02의 가위바위보 손 그림은 같은 손 모델의 그림끼리 거의 같아서 손 모델 단위로 나눕니다. 이 교안의 MNIST 표본에는 쓴 사람 정보가 없어 그림 단위로 무작위로 나눕니다.


<img src="images/77_leakage_group_split.png" width="1050" alt="제목은 '데이터 누수와 묶음 단위 분할'입니다. 두 판 모두 위쪽에 숫자 카드 9장(배지 A가 붙은 3, 7, 1, 배지 B가 붙은 3, 7, 1, 배지 C가 붙은 3, 7, 1)이 있고, 화살표가 파란 '학습' 바구니와 빨간 '검증' 바구니로 갈라집니다. 왼쪽 '그림 단위로 섞어 나누기' 판은 학습에 A의 3과 1, B의 3과 7, C의 7과 1 여섯 장, 검증에 A의 7, B의 1, C의 3 세 장이 들어가고, 학습의 A 카드와 검증의 A 카드가 빨간 점선 화살표로 이어지며 '검증 정확도가 실제보다 높게 나옴' 칩이 붙어 있습니다. 오른쪽 '묶음 단위 분할' 판은 학습에 A와 B의 여섯 장, 검증에 C의 세 장이 들어가고 '검증 = 처음 보는 사람의 글씨' 칩이 붙어 있습니다. 맨 아래 띠에 '같은 사람 · 같은 원본에서 나온 데이터는 한쪽에만 둡니다'가 적혀 있습니다.">


1. 학습용 10,000장의 순서를 시드 0으로 섞고 앞 1,000장만 씁니다. 그림이 적어야 과적합이 잘 보이고, CPU에서도 한 에폭이 1초 안에 끝납니다.
2. 그 가운데 앞 800장은 학습 데이터, 다음 200장은 검증 데이터입니다.
3. 테스트용 2,000장은 그대로 테스트 데이터입니다. 검증 손실로 고른 모델 하나에만 씁니다.

In [ ]:
# 픽셀을 0~1 실수로 바꾸고 채널 축을 넣어 (그림 수, 1, 28, 28) 모양으로 만듭니다
images = (mnist["train_images"].float() / 255).unsqueeze(1)
labels = mnist["train_labels"]

# 학습용 10,000장의 순서를 섞습니다. 같은 시드면 언제나 같은 순서입니다
generator = torch.Generator().manual_seed(0)
order = torch.randperm(len(labels), generator=generator)

# 섞은 순서의 앞 800장은 학습, 다음 200장은 검증
X_train = images[order[:800]]
y_train = labels[order[:800]]
X_valid = images[order[800:1000]]
y_valid = labels[order[800:1000]]

# 테스트 2,000장은 마지막에 한 번만 씁니다
X_test = (mnist["test_images"].float() / 255).unsqueeze(1)
y_test = mnist["test_labels"]
print("학습:", X_train.shape, "| 검증:", X_valid.shape, "| 테스트:", X_test.shape)

In [ ]:
# 숫자(0~9)마다 학습 그림과 검증 그림이 몇 장인지 셉니다. 열 이름 0~9가 숫자입니다
count_table = pd.DataFrame({"학습": torch.bincount(y_train, minlength=10).tolist(),
                            "검증": torch.bincount(y_valid, minlength=10).tolist()})
display(count_table.T)

**결과 읽기:** 숫자마다 학습 그림이 60~102장, 검증 그림이 15~27장입니다. 교안 01은 숫자마다 약 1,000장으로 학습했으니 약 10분의 1인 양입니다.

### 1-3. DataLoader와 학습 함수

미니배치 크기(64), 섞는 방법, 학습용의 `drop_last=True`는 교안 01(4-2)과 같습니다. 800장을 64장씩 나누면 남는 32장을 버리므로 학습 미니배치는 12개입니다. 검증용과 테스트용은 결과만 계산하므로 섞지 않습니다. 한 에폭 학습·평가 함수는 지난 시간에 만든 두 함수를 그대로 씁니다(Dropout을 넣을 때 한 줄씩 더합니다).

In [ ]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# 학습용은 에폭마다 섞고, 64장이 안 되는 마지막 미니배치는 버립니다(교안 01 4-2)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True, drop_last=True)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=64, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=64, shuffle=False)
print("학습 미니배치 수:", len(train_loader), "| 검증:", len(valid_loader), "| 테스트:", len(test_loader))

# 출력 10개 + CrossEntropyLoss(지난 시간 다중 분류의 짝)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None):
    """미니배치마다 파라미터를 한 번씩 갱신하고, 그 에폭 미니배치 손실의 평균을 돌려줍니다."""
    total_loss = 0.0
    for X_batch, y_batch in loader:
        # 미니배치를 모델이 있는 계산 장치로 옮깁니다
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # 지난 스텝의 기울기를 지우고, 순전파 -> 손실 -> 역전파 -> 갱신
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

        # 스케줄러가 있으면 스텝마다 학습률을 바꿉니다
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()

    # len(loader)는 미니배치 수입니다
    return total_loss / len(loader)


def evaluate(model, loader, loss_fn):
    """기울기 없이 예측해 (평균 손실, 정확도)를 돌려줍니다."""
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(X_batch)

            # 마지막 미니배치는 장 수가 적으므로, 손실에 장 수를 곱해 더한 뒤 전체 장 수로 나눕니다
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()

    n_samples = len(loader.dataset)
    return total_loss / n_samples, correct / n_samples

## 2. 과적합 진단

과적합은 테스트 정확도 하나로는 보이지 않으므로, 에폭마다 학습 손실과 검증 손실을 함께 기록해 두 곡선의 모양으로 진단합니다.

### 2-1. 학습 곡선의 대표적인 세 모양

**학습 곡선(learning curve):** 가로축이 에폭, 세로축이 손실인 그래프에 학습 손실과 검증 손실을 함께 그린 것입니다. 보통 검증 손실이 학습 손실보다 위에 놓입니다(Dropout처럼 학습할 때만 켜는 규제가 있으면 반대가 되기도 합니다. 8-5 참고). 두 곡선의 움직임으로 모델 상태를 대표적인 세 가지로 나눕니다.


<img src="images/72_fit_shapes.png" width="1050" alt="제목은 '학습 곡선의 대표적인 세 모양'입니다. 판 세 개 모두 가로축 '에폭', 세로축 '손실'에 파란 '학습 손실'과 빨간 '검증 손실'이 있고, 세 판 모두 검증 손실이 학습 손실보다 위에 있습니다. '과소적합' 판은 두 선이 조금 내려가다 높은 곳에서 평평해지고 아래에 '둘 다 높음: 모델을 키우거나 더 학습'이 적혀 있습니다. '적절' 판은 두 선이 함께 내려가 낮은 곳에서 나란히 평평해지고 아래에 '둘 다 낮고 가까움'이 적혀 있습니다. '과적합' 판은 학습 손실이 바닥 가까이 계속 내려가고 검증 손실은 가운데쯤 가장 낮은 점(별)에서 다시 올라가며, 아래에 '학습 손실만 줄고 검증 손실은 다시 커짐: 규제, 조기 종료'가 적혀 있습니다.">


| 학습 손실 | 검증 손실 | 진단 | 먼저 해 볼 것 |
|---|---|---|---|
| 높음 | 높음 | 과소적합(underfitting): 학습 데이터조차 충분히 맞히지 못합니다 | 모델 키우기, 더 오래 학습, 학습률 다시 고르기 |
| 낮음 | 낮고 학습 손실과 가까움 | 적절 | 그대로 쓰기 |
| 계속 줄어듦 | 가장 낮은 에폭 뒤 다시 커짐 | 과적합: 그 에폭 뒤로는 학습 데이터에만 맞춥니다 | 규제, 조기 종료 |

### 2-2. 큰 CNN 만들기

과적합이 잘 보이도록 교안 01의 CNN보다 큰 모델을 만듭니다.

- **필터 수:** 합성곱 묶음 두 개의 필터를 16, 32개에서 32, 64개로 두 배씩 늘립니다.
- **은닉층:** 펼친 뒤 바로 출력층으로 가지 않고, 은닉층 뉴런 128개(ReLU)를 한 번 거칩니다.
- **모양 변화:** (1, 28, 28) → (32, 14, 14) → (64, 7, 7) → 펼쳐서 3,136개 → 128개 → 로짓(logit) 10개입니다. 로짓은 출력층이 종류마다 내는, softmax를 거치기 전의 값입니다.

In [ ]:
def make_big_cnn():
    """필터 32개, 64개인 합성곱 묶음 두 개와 은닉층 128개로 된 CNN을 만들어 계산 장치에 올려 돌려줍니다."""
    return nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # (32, 14, 14)
        nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # (64, 7, 7)
        nn.Flatten(),                                                             # 3,136개 한 줄
        nn.Linear(64 * 7 * 7, 128), nn.ReLU(),                                    # 은닉층 뉴런 128개
        nn.Linear(128, 10),                                                       # 로짓 10개
    ).to(device)


# 파라미터 수를 학습 그림 수와 비교합니다
torch.manual_seed(0)
big_model = make_big_cnn()
n_params = 0
for param in big_model.parameters():
    n_params += param.numel()
print("파라미터 수:", n_params, "| 학습 그림 수:", len(X_train))

**결과 읽기:** 파라미터가 421,642개로 학습 그림 800장의 500배가 넘습니다. 이렇게 파라미터가 많으면 그림 800장의 정답을 하나하나 외울 수 있어 과적합이 잘 생깁니다.

### 2-3. 학습 곡선 그리기

에폭마다 학습한 뒤 검증 데이터로 평가해 기록하는 일을 뒤에서 여러 번 하므로 함수 `fit`으로 둡니다.

- **학습 손실:** `train_one_epoch`가 돌려준 그 에폭 미니배치 손실의 평균입니다.
- **검증 손실, 검증 정확도:** 에폭이 끝난 모델로 `evaluate(model, valid_loader, loss_fn)`을 불러 계산합니다.
- **설정:** Adam 학습률 0.001, 30에폭입니다(12스텝 × 30 = 360번 갱신).
- **실행 환경:** 계산 장치와 스레드 수에 따라 소수 셋째 자리와 멈춘 에폭이 달라지고, 테스트 정확도도 0.01 안팎 달라질 수 있습니다. 결론은 곡선의 모양과 큰 차이로 읽습니다.

In [ ]:
def fit(model, optimizer, epochs):
    """train_loader로 epochs 에폭 학습하며, 에폭마다 학습 손실과 검증 손실, 검증 정확도를 기록한 표를 돌려줍니다."""
    rows = []
    for epoch in range(1, epochs + 1):
        # 한 에폭 학습한 뒤 검증 데이터로 평가해 표의 한 행으로 모읍니다
        train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
        valid_loss, valid_accuracy = evaluate(model, valid_loader, loss_fn)
        rows.append({"에폭": epoch, "학습 손실": train_loss, "검증 손실": valid_loss, "검증 정확도": valid_accuracy})
    return pd.DataFrame(rows)

In [ ]:
# 같은 초깃값에서 시작하도록 시드를 고정하고, 큰 CNN을 30에폭 학습합니다
torch.manual_seed(0)  # 모델 초깃값과 미니배치 섞는 순서를 고정합니다
big_model = make_big_cnn()
big_optimizer = torch.optim.Adam(big_model.parameters(), lr=0.001)
big_history = fit(big_model, big_optimizer, 30)

# 검증 손실이 가장 낮은 에폭과 마지막 에폭을 나란히 봅니다
best_index = big_history["검증 손실"].idxmin()
display(big_history.loc[[best_index, len(big_history) - 1]].set_index("에폭").round(4))

In [ ]:
# 왼쪽: 학습 손실과 검증 손실, 오른쪽: 검증 정확도. 점선은 검증 손실이 가장 낮은 에폭입니다
best_epoch = big_history.loc[best_index, "에폭"]
plt.figure(figsize=(14, 4.5))
plt.subplot(1, 2, 1)
plt.plot(big_history["에폭"], big_history["학습 손실"], linewidth=2, label="학습 손실")
plt.plot(big_history["에폭"], big_history["검증 손실"], linewidth=2, label="검증 손실")
plt.axvline(best_epoch, color="gray", linestyle="--")
plt.ylim(0, 1.0)  # 첫 에폭의 큰 손실은 잘라 뒤쪽 차이가 보이게 합니다
plt.xlabel("에폭", fontsize=12)
plt.ylabel("손실", fontsize=12)
plt.title("큰 CNN의 학습 곡선", fontsize=14)
plt.legend(fontsize=11)
plt.subplot(1, 2, 2)
plt.plot(big_history["에폭"], big_history["검증 정확도"], linewidth=2, color="green")
plt.axvline(best_epoch, color="gray", linestyle="--")
plt.xlabel("에폭", fontsize=12)
plt.ylabel("검증 정확도", fontsize=12)
plt.title("검증 정확도", fontsize=14)
plt.tight_layout()
plt.show()

**결과 읽기**
- **학습 손실:** 30에폭에 0.0024까지 내려갑니다. 학습 그림 800장을 거의 다 외웠습니다.
- **검증 손실:** 10에폭에 0.1685로 가장 낮고, 그 뒤로 다시 커져 30에폭에 0.2360입니다. 학습 곡선의 세 모양 가운데 과적합입니다.
- **검증 정확도:** 10에폭 0.95에서 30에폭 0.94로 손실만큼 크게 변하지 않습니다. 손실이 커진 이유는 조기 종료를 배운 뒤 6-3에서 두 모델을 비교해 확인합니다.
- 이어서 다시 커지는 검증 손실을 규제로 줄이고, 조기 종료로 가장 낮은 에폭에서 멈춥니다.

**에폭 간 변화**
- 검증 손실은 에폭마다 조금씩 오르내리므로, 설정끼리 최저 검증 손실을 비교할 때는 차이가 이 오르내림보다 큰지 확인해야 합니다.
- 15에폭 뒤 구간에서 이웃한 두 에폭의 검증 손실 차이(절댓값)를 평균한 값을 **에폭 간 변화**로 부릅니다. 앞쪽 에폭은 손실이 크게 내려가는 중이라 뺍니다.

In [ ]:
def epoch_change(history):
    """15에폭 뒤 구간에서 이웃한 두 에폭의 검증 손실 차이(절댓값)의 평균을 돌려줍니다."""
    # iloc[14:]는 15번째 행(15에폭)부터입니다. diff로 이웃한 에폭의 차이를 구해 절댓값을 평균합니다
    return history["검증 손실"].iloc[14:].diff().abs().mean()


print("큰 CNN의 에폭 간 변화:", round(epoch_change(big_history), 4))

**결과 읽기:** 큰 CNN의 에폭 간 변화는 0.0071입니다. 뒤에서 설정끼리 최저 검증 손실을 비교할 때 이 값을 기준으로 삼습니다. 최저 검증 손실의 차이가 이보다 작으면 설정의 차이인지 오르내림인지 가를 수 없습니다.

## 3. Dropout

모델이 몇몇 뉴런의 특정 조합에 기대어 학습 그림을 외우지 못하게, 학습할 때마다 뉴런 일부를 무작위로 뺍니다.

### 3-1. Dropout의 작동

**Dropout:** 학습할 때 층의 출력(뉴런 값)을 원소마다 확률 `p`로 0으로 만드는 층입니다([Srivastava et al., 2014](https://jmlr.org/papers/v15/srivastava14a.html)). 순전파할 때마다 0이 되는 뉴런이 새로 정해집니다.

- **학습 모드(`model.train()`):** 원소마다 확률 `p`로 0을 만들고, 남은 값은 $\frac{1}{1 - p}$배로 키웁니다. 키우는 이유는 평가할 때 모든 뉴런이 켜져도 다음 층에 들어가는 값의 평균이 학습 때와 같게 하려는 것입니다.
- **평가 모드(`model.eval()`):** 아무것도 0으로 만들지 않고 그대로 통과시킵니다. 예측은 늘 같은 결과여야 하기 때문입니다.


<img src="images/73_dropout.png" width="1050" alt="제목은 'Dropout: 학습할 때만 일부 뉴런을 0으로 만듭니다'입니다. 두 판에 '입력 3개', '은닉 뉴런 6개', '출력 1개'인 같은 신경망이 있습니다. 왼쪽 '학습 모드 model.train()' 판은 은닉 뉴런 6개 가운데 3개가 회색 X이고 그 뉴런에서 출력으로 가는 선이 회색 점선이며, 설명 상자에 '뉴런마다 확률 p로 0', '남은 뉴런은 1 / (1 - p)배', '순전파마다 0이 되는 뉴런이 새로 정해짐'이 적혀 있습니다. 오른쪽 '평가 모드 model.eval()' 판은 뉴런 6개가 모두 켜져 있고 설명 상자에 '모든 뉴런을 그대로 통과'가 적혀 있습니다.">


아래 움직이는 그림은 같은 은닉 뉴런 6개에 `p = 0.5`인 Dropout을 학습 모드로 다섯 번, 평가 모드로 한 번 넣은 결과입니다. 0이 되는 뉴런이 순전파마다 바뀌고(개수도 달라질 수 있습니다), 남은 값은 늘 넣은 값의 2배입니다.

<img src="images/7a_dropout_passes.gif" width="900" alt="움직이는 그림입니다. 제목은 'Dropout: 순전파마다 0이 되는 뉴런이 새로 정해집니다'입니다. 왼쪽에 입력 3개, 은닉 뉴런 6개, 출력 1개인 신경망이 있고, 오른쪽 표 첫 행 '넣은 값'은 0.6, 1.5, 0.3, 1.1, 0.9, 0.4입니다. 칩 '학습 모드 model.train(), p = 0.5' 아래에서 순전파 1부터 5까지 한 장씩 지나가며 표에 한 행씩 쌓입니다. 행마다 0이 된 뉴런은 회색 0.0(신경망에서는 회색 X와 점선)이고 남은 뉴런은 넣은 값의 2배이며, 0의 개수는 순전파마다 2개 또는 3개로 다릅니다. 아래 띠에 '순전파 1: 뉴런 3개가 0, 남은 3개는 1 / (1 - 0.5) = 2배'처럼 적힙니다. 마지막 장은 칩이 '평가 모드 model.eval()'로 바뀌고 뉴런 6개가 모두 켜지며, 표 마지막 행 '평가 모드'가 넣은 값과 같고, 띠에 '평가 모드: 아무것도 0으로 만들지 않고 그대로 통과합니다'가 적혀 있습니다.">

`p = 0.5`인 Dropout에 값이 모두 1인 원소 10개를 넣으면 평균적으로 다음과 같습니다. 실제로 0이 되는 개수는 실행마다 다릅니다.

1. **학습 모드:** 10개 가운데 평균 5개가 0이 되고, 남은 값은 1 / (1 - 0.5) = 2가 됩니다.
2. **평균:** 0이 5개, 2가 5개면 평균은 1로, 넣은 값의 평균과 같습니다.
3. **평가 모드:** 10개 모두 1 그대로입니다.

In [ ]:
# p=0.5인 Dropout에 값이 모두 1인 텐서를 넣어 봅니다
torch.manual_seed(0)
dropout = nn.Dropout(p=0.5)
ones = torch.ones(10)

# 학습 모드: 원소마다 확률 p(0.5)로 0, 남은 값은 1 / (1 - p) = 2배
dropout.train()
train_output = dropout(ones)
print("학습 모드:", train_output, "| 평균:", round(train_output.mean().item(), 4))

# 평가 모드: 그대로 통과
dropout.eval()
print("평가 모드:", dropout(ones))

**결과 읽기:** 학습 모드에서는 이번에 6개가 0이 되고 4개가 2.0이 되어 평균이 0.8입니다. 0이 되는 개수는 실행마다 다르고 평균하면 5개입니다. 평가 모드는 10개 모두 1.0입니다.

### 3-2. 학습 모드와 평가 모드를 바꾸는 학습 함수

지난 시간의 `train_one_epoch`와 `evaluate`는 모델의 모드를 바꾸지 않습니다. 모델은 만들 때 학습 모드로 시작하므로, 그대로 쓰면 검증 손실을 계산할 때도 Dropout이 뉴런 출력을 0으로 만듭니다. 그래서 두 함수의 맨 앞에 한 줄씩 더합니다.

- **`train_one_epoch`의 첫 줄 `model.train()`:** 학습하는 동안 Dropout이 작동하게 합니다. 앞 에폭 끝의 `evaluate`가 평가 모드로 바꿔 두므로, 에폭마다 다시 학습 모드로 돌려야 합니다.
- **`evaluate`의 첫 줄 `model.eval()`:** 평가하는 동안 Dropout이 아무것도 0으로 만들지 않게 합니다.
- Dropout이 없는 모델(큰 CNN)에서는 두 줄이 있어도 결과가 같습니다. 이름과 인자가 그대로이므로 `fit`은 고치지 않고 새 함수를 씁니다.

`evaluate`는 `model.eval()`과 `torch.no_grad()`를 함께 씁니다. 둘은 하는 일이 달라 하나가 다른 하나를 대신하지 못합니다.

| | 하는 일 | 빼먹으면 |
|---|---|---|
| `model.eval()` | 모델 안 층의 동작을 평가용으로 바꿉니다. Dropout은 아무것도 0으로 만들지 않고, 배치 정규화(교안 01 5-2)는 미니배치 통계 대신 모아 둔 이동 평균을 씁니다 | 검증 손실이 학습 모드로 계산됩니다. Dropout은 실행마다 값이 바뀌고(3-4에서 확인), 배치 정규화는 이동 평균이 검증 데이터로 바뀝니다 |
| `torch.no_grad()` | 블록 안의 계산을 역전파용으로 기록하지 않습니다. 층의 동작은 바꾸지 않습니다 | 예측 값은 같지만, 역전파에 쓸 중간 결과를 저장하느라 메모리와 시간을 더 씁니다 |

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None):
    """미니배치마다 파라미터를 한 번씩 갱신하고, 그 에폭 미니배치 손실의 평균을 돌려줍니다."""
    model.train()  # 학습 모드: Dropout이 뉴런 출력 일부를 0으로 만듭니다
    total_loss = 0.0
    for X_batch, y_batch in loader:
        # 미니배치를 모델이 있는 계산 장치로 옮깁니다
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # 지난 스텝의 기울기를 지우고, 순전파 -> 손실 -> 역전파 -> 갱신
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

        # 스케줄러가 있으면 스텝마다 학습률을 바꿉니다
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()

    # len(loader)는 미니배치 수입니다
    return total_loss / len(loader)


def evaluate(model, loader, loss_fn):
    """기울기 없이 예측해 (평균 손실, 정확도)를 돌려줍니다."""
    model.eval()  # 평가 모드: Dropout이 아무것도 0으로 만들지 않습니다
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(X_batch)

            # 마지막 미니배치는 장 수가 적으므로, 손실에 장 수를 곱해 더한 뒤 전체 장 수로 나눕니다
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()

    n_samples = len(loader.dataset)
    return total_loss / n_samples, correct / n_samples

### 3-3. Dropout을 넣은 CNN 학습하기

큰 CNN과 Dropout만 다른 모델을 같은 시드로 학습해 비교합니다.

- **넣는 자리:** 파라미터가 가장 많은 곳(펼친 3,136개가 은닉층 뉴런 128개로 가는 층)의 앞과, 은닉층 ReLU 뒤 두 곳입니다.
- **`p`:** 0.1~0.5를 많이 씁니다. 여기서는 효과가 잘 보이는 0.5를 씁니다. 너무 크면 학습 데이터도 맞히지 못하는 과소적합이 될 수 있습니다.
- **`p = 0`이면:** 아무것도 0으로 만들지 않으므로 큰 CNN과 같은 모델입니다. 뒤에서 다섯 설정을 비교할 때 이 점을 씁니다.

In [ ]:
def make_dropout_cnn(dropout_p):
    """make_big_cnn과 같은 구조에, 펼친 뒤와 은닉층 뒤에 Dropout(dropout_p)을 넣은 CNN을 돌려줍니다."""
    return nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Dropout(dropout_p),
        nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(dropout_p),
        nn.Linear(128, 10),
    ).to(device)


# 큰 CNN과 같은 시드, 같은 옵티마이저로 30에폭 학습합니다
torch.manual_seed(0)
drop_model = make_dropout_cnn(0.5)
drop_optimizer = torch.optim.Adam(drop_model.parameters(), lr=0.001)
drop_history = fit(drop_model, drop_optimizer, 30)

설정 여러 개의 학습 곡선과 요약 표를 뒤에서도 계속 비교하므로, 그리는 함수와 요약하는 함수를 둡니다.

In [ ]:
def plot_histories(histories):
    """설정마다 학습 손실(왼쪽)과 검증 손실(오른쪽) 곡선을 겹쳐 그립니다. histories는 {설정 이름: fit이 돌려준 표}입니다."""
    plt.figure(figsize=(14, 4.5))
    for k, column in enumerate(["학습 손실", "검증 손실"]):
        plt.subplot(1, 2, k + 1)
        for name in histories:
            plt.plot(histories[name]["에폭"], histories[name][column], linewidth=2, label=name)
        plt.ylim(0, 0.8)  # 처음 몇 에폭의 큰 손실은 잘라 뒤쪽 차이가 보이게 합니다
        plt.xlabel("에폭", fontsize=12)
        plt.ylabel(column, fontsize=12)
        plt.title(column, fontsize=14)
        plt.legend(fontsize=11, ncol=2, loc="upper right")
    plt.tight_layout()
    plt.show()


def summarize(histories):
    """설정마다 가장 낮은 검증 손실과 그 에폭, 마지막 에폭의 학습 손실과 검증 손실, 에폭 간 변화를 한 행으로 모은 표를 돌려줍니다."""
    rows = []
    for name in histories:
        history = histories[name]
        best = history.loc[history["검증 손실"].idxmin()]
        last = history.iloc[-1]
        rows.append({"설정": name, "최저 검증 손실": best["검증 손실"], "그 에폭": int(best["에폭"]),
                     "마지막 학습 손실": last["학습 손실"], "마지막 검증 손실": last["검증 손실"],
                     "에폭 간 변화": epoch_change(history)})
    return pd.DataFrame(rows)


drop_histories = {"규제 없음": big_history, "Dropout 0.5": drop_history}
plot_histories(drop_histories)
display(summarize(drop_histories).round(4))

**결과 읽기**
- **학습 손실:** Dropout 모델이 더 높습니다(30에폭 0.0802, 규제 없음 0.0024). 학습할 때 뉴런 출력의 절반(p = 0.5)이 0인 채로 계산하므로 학습 그림을 외우기 어렵습니다.
- **검증 손실:** 가장 낮은 값이 0.1685(10에폭)에서 0.1179(21에폭)로 낮아졌고, 30에폭에도 0.1417로 규제 없음(0.2360)처럼 크게 오르지 않습니다.

### 3-4. 평가 모드를 빼먹을 때의 검증 손실

`evaluate`의 `model.eval()` 한 줄이 왜 필요한지, 학습을 마친 Dropout 모델에 검증 그림 200장을 두 모드로 넣어 손실을 비교합니다.

In [ ]:
# 검증 그림 200장과 정답을 계산 장치로 옮겨 둡니다
X_valid_device = X_valid.to(device)
y_valid_device = y_valid.to(device)

# 학습 모드로 두고 같은 검증 그림을 두 번 넣습니다
drop_model.train()
with torch.no_grad():
    train_mode_loss_1 = loss_fn(drop_model(X_valid_device), y_valid_device).item()
    train_mode_loss_2 = loss_fn(drop_model(X_valid_device), y_valid_device).item()

# 평가 모드로 바꾸고 같은 검증 그림을 두 번 넣습니다
drop_model.eval()
with torch.no_grad():
    eval_mode_loss_1 = loss_fn(drop_model(X_valid_device), y_valid_device).item()
    eval_mode_loss_2 = loss_fn(drop_model(X_valid_device), y_valid_device).item()

print(f"학습 모드 두 번: {train_mode_loss_1:.4f}, {train_mode_loss_2:.4f}")
print(f"평가 모드 두 번: {eval_mode_loss_1:.4f}, {eval_mode_loss_2:.4f}")

**결과 읽기**
- **학습 모드:** 같은 그림인데 두 손실이 서로 다르고, 둘 다 평가 모드보다 큽니다. 넣을 때마다 0이 되는 뉴런이 바뀌기 때문입니다.
- **평가 모드:** 두 번 모두 0.1417로 같고, Dropout 요약 표의 마지막 검증 손실과도 같습니다.
- Dropout이 있는 모델에서 `model.eval()`을 빼먹으면 검증 손실이 평가 모드의 값과 다르고 실행마다 바뀌어 설정을 잘못 고를 수 있습니다.

## 4. Weight decay

과적합된 모델은 학습 그림의 작은 차이까지 맞추려고 일부 가중치를 크게 키우기 쉬우므로, 갱신할 때마다 가중치를 0 쪽으로 조금씩 줄여 커지지 못하게 합니다.

### 4-1. AdamW가 가중치를 줄이는 방식

**가중치 감쇠(weight decay):** 갱신할 때마다 가중치를 0 쪽으로 일정 비율 줄이는 규제입니다. AdamW는 파라미터 $w$를 아래 식으로 줄인 뒤, 기울기로 하는 갱신을 그대로 합니다. 학습률 $\eta$에 $\lambda$(람다)를 곱한 비율만큼 줄입니다. $\lambda$가 `weight_decay` 인자이고 클수록 세게 줄입니다.

$\displaystyle w \leftarrow w - \eta \, \lambda \, w$

- 손실을 줄이는 데 꼭 필요한 가중치만 기울기 덕분에 큰 값을 유지하고, 나머지는 0 쪽으로 줄어듭니다.
- 지난 시간에 실무 기본 옵티마이저로 본 AdamW는 Adam에 이 방식의 weight decay를 더한 옵티마이저입니다. 지난 시간의 5에폭 비교에서 Adam과 결과가 거의 같았던 이유는 아래 예의 3번과 같습니다. 기본값 $\lambda$ = 0.01은 학습률 0.001에서 한 번에 0.001%씩만 줄이므로 짧은 학습에서는 차이가 거의 없습니다.
- PyTorch에서는 `torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=λ)`로 씁니다([Loshchilov & Hutter, 2019](https://arxiv.org/abs/1711.05101), [AdamW 문서](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html)). `weight_decay`를 적지 않으면 0.01입니다.
- `torch.optim.Adam`에도 `weight_decay` 인자가 있지만, 기울기에 $\lambda w$를 더하는 방식(L2 규제)이라 $\lambda w$도 Adam의 파라미터별 크기 조절을 받아, 기울기가 컸던 파라미터일수록 덜 줄어듭니다. AdamW는 기울기와 따로, 갱신 단계에서 가중치를 직접 줄입니다(decoupled weight decay). 그래서 Adam 계열로 weight decay를 걸 때는 AdamW를 씁니다.


<img src="images/74_weight_decay.png" width="1050" alt="제목은 'Weight decay: 갱신할 때마다 가중치를 조금씩 줄입니다'입니다. 왼쪽 막대그래프 '30에폭 뒤 가중치 크기(절편 제외 L2 노름)'는 4-2에서 측정한 값으로, '규제 없음' 14.6(회색), 'λ = 0.01' 14.5, 'λ = 1.0' 12.2, 'λ = 10.0' 8.0(초록)이고, λ = 10.0 막대에 '너무 크면 과소적합' 말풍선이 붙어 있습니다. 오른쪽에 카드 두 장 '1 가중치 줄이기: w - 학습률 × λ × w'와 '2 Adam 갱신: 기울기로 이동'이 위에서 아래로 화살표로 이어집니다. 아래 띠 두 줄에 '예: w = 2.0, 학습률 0.001, λ = 1.0 -> 2.0 - 0.001 × 1.0 × 2.0 = 1.998'과 'λ가 클수록 가중치가 작아집니다. λ는 검증 손실로 고릅니다'가 적혀 있습니다.">


가중치 하나($w$ = 2.0)를 학습률 0.001로 갱신하는 예입니다.

1. **λ = 1.0:** 한 번 갱신하면 $w$ = 2.0 - 0.001 × 1.0 × 2.0 = 1.998입니다. 0.002, 곧 0.1%씩 줄어듭니다.
2. **360번 갱신(30에폭):** 기울기가 다시 키우지 않으면 0.999를 360번 곱해 원래의 약 70%가 됩니다.
3. **λ = 0.01(기본값):** 한 번에 0.001%씩 줄어, 360번 갱신해도 원래의 약 99.6%로 거의 그대로입니다.

### 4-2. 규제 세기 λ 비교

λ가 너무 작으면 효과가 없고 너무 크면 과소적합이 되므로, 기본값 0.01과 1.0, 10.0으로 같은 큰 CNN을 30에폭씩 학습해 비교합니다.

- **가중치 크기:** 절편(bias)을 뺀 모든 층 가중치의 제곱을 더해 제곱근을 씌운 값(L2 노름)입니다. 가중치가 전체적으로 클수록 커집니다. 이 교안처럼 `model.parameters()`를 모두 넘기면 절편에도 weight decay가 걸리지만, 가중치 크기에서는 뺍니다.

$\displaystyle \sqrt{\sum w^2}$ (가중치가 3과 4 두 개면 $\sqrt{9 + 16} = 5$)

- **고르는 기준:** 검증 손실입니다. 테스트 데이터는 쓰지 않습니다.

In [ ]:
def weight_size(model):
    """층 가중치(절편 제외)의 제곱을 모두 더해 제곱근을 씌운 값을 돌려줍니다."""
    total = 0.0
    for name, param in model.named_parameters():
        if name.endswith("weight"):
            total += (param ** 2).sum().item()
    return total ** 0.5


# weight_decay 값마다 같은 시드로 큰 CNN을 새로 만들어 학습합니다. 옵티마이저만 AdamW로 바꿉니다
decay_histories = {"규제 없음": big_history}
decay_sizes = {"규제 없음": weight_size(big_model)}
for weight_decay in [0.01, 1.0, 10.0]:
    torch.manual_seed(0)
    decay_model = make_big_cnn()
    decay_optimizer = torch.optim.AdamW(decay_model.parameters(), lr=0.001, weight_decay=weight_decay)
    name = f"weight decay {weight_decay}"
    decay_histories[name] = fit(decay_model, decay_optimizer, 30)
    decay_sizes[name] = weight_size(decay_model)

In [ ]:
# 요약 표에 30에폭 뒤 가중치 크기 열을 더합니다
decay_table = summarize(decay_histories)
decay_table["가중치 크기"] = list(decay_sizes.values())
display(decay_table.round(4))
plot_histories(decay_histories)

**결과 읽기**
- **가중치 크기:** 규제 없음 약 14.6, λ 0.01도 약 14.5로 거의 같습니다. 기본값 0.01은 이 학습(360번 갱신)에서는 너무 약합니다. λ 1.0은 약 12.2, 10.0은 약 8.0으로 작아집니다.
- **λ 0.01:** 곡선이 규제 없음과 거의 겹칩니다. 가장 낮은 검증 손실(0.1684, 규제 없음 0.1685)의 차이 0.0001은 에폭 간 변화(표의 마지막에서 두 번째 열, 0.0078과 0.0071)보다 작습니다.
- **λ 1.0:** 가장 낮은 검증 손실(0.1752)은 규제 없음과 차이 0.007로 에폭 간 변화(0.0107)보다 작지만, 30에폭 검증 손실이 0.2035로 규제 없음(0.2360)보다 덜 오릅니다. 다시 커지는 속도를 늦췄습니다.
- **λ 10.0:** 학습 손실이 30에폭에도 0.3069로 높고, 검증 손실도 가장 낮을 때가 0.3046입니다. 너무 세게 줄여 과소적합이 됐습니다.
- 이 교안의 모델에서는 Dropout보다 효과가 작습니다. λ에 따라 효과가 크게 달라지므로 늘 검증 손실로 확인해 고릅니다.
- **더 볼 후보는 λ 1.0:** 가장 낮은 검증 손실은 규제 없음, λ 0.01, λ 1.0이 에폭 간 변화 안에서 비슷합니다. λ 1.0만 그 뒤 다시 커지는 속도가 느리므로, 조기 종료로 더 오래 학습하는 비교에서 λ 1.0을 확인합니다.

## 5. 데이터 증강

학습 그림이 적을수록 과적합이 잘 생기므로, 모델을 바꾸는 대신 학습 그림을 조금씩 바꿔 매 에폭 새 그림처럼 보이게 합니다.

### 5-1. 그림을 옮기는 증강 함수

**데이터 증강(data augmentation):** 정답이 바뀌지 않는 범위에서 학습 그림을 조금 바꿔(옮기기, 뒤집기, 돌리기) 학습 데이터가 늘어난 효과를 내는 방법입니다. 검증·테스트 그림은 바꾸지 않습니다.

자주 쓰는 그림 증강의 예는 아래와 같고, 실무에서는 `torchvision.transforms.v2`의 변환으로 씁니다. 이 단원 실습 환경에는 torchvision이 없으므로, 이 교안은 옮기기만 `torch.roll`로 직접 만듭니다.

| 증강 | `torchvision.transforms.v2` 예 | 하는 일 | 쓰지 않는 경우 |
|---|---|---|---|
| 좌우 뒤집기 | `RandomHorizontalFlip()` | 확률 0.5로 좌우를 뒤집습니다 | 좌우가 바뀌면 다른 뜻이 되는 그림(숫자, 글자) |
| 회전 | `RandomRotation(degrees=15)` | -15°~15° 사이 각도로 돌립니다 | 크게 돌리면 정답이 바뀌는 그림(6을 180° 돌리면 9) |
| 이동 | `RandomAffine(degrees=0, translate=(0.1, 0.1))` | 가로, 세로로 그림 크기의 10% 안에서 옮깁니다 | 물체가 가장자리에 붙어 있어 옮기면 일부가 잘릴 때 |
| 자르기·크기 조절 | `RandomResizedCrop(size=28, scale=(0.6, 1.0))` | 넓이 60~100%인 구역을 잘라 원래 크기로 늘립니다 | 잘린 구역에서 정답을 가르는 부분이 빠질 때 |
| 밝기·대비 | `ColorJitter(brightness=0.3, contrast=0.3)` | 밝기와 대비를 0.7~1.3배 사이로 바꿉니다 | 밝기나 색 자체가 정답을 가를 때(덜 익은 과일과 익은 과일) |
| 잡음 | `GaussianNoise(sigma=0.1)` | 픽셀마다 평균 0, 표준편차 0.1인 잡음을 더합니다 | 실제 입력에 없는 만큼 큰 잡음 |


<img src="images/78_augment_types.png" width="1050" alt="제목은 '그림 증강의 종류: 정답이 바뀌지 않는 변환만 씁니다'입니다. 열 이름은 왼쪽부터 '원래 그림', '좌우 뒤집기', '회전', '이동', '자르기·크기 조절', '밝기·대비', '잡음'입니다. 윗줄 '손 그림(가위)'은 과제 02의 가위 손 그림 한 장에 여섯 변환을 적용한 것으로, 여섯 칸 모두 초록 테두리에 '씀'(회전 칸은 '15°: 씀')이 적혀 있습니다. 아랫줄 '숫자 6'은 MNIST 학습 그림의 6 한 장으로, 좌우로 뒤집은 칸은 빨간 테두리에 '다른 모양', 180° 돌린 칸은 9처럼 보이며 빨간 테두리에 '180°: 9처럼 보임', 나머지 네 칸은 초록 테두리에 '씀'이 적혀 있습니다. 맨 아래 띠에 '숫자에는 좌우 뒤집기와 큰 각도 회전을 쓰지 않습니다'가 적혀 있습니다.">


- **옮기기:** 숫자 7을 오른쪽으로 2픽셀 옮겨도 여전히 7입니다. 손으로 쓸 때마다 위치가 조금씩 다르므로, 옮긴 그림은 실제로 있을 법한 그림입니다.
- **`torch.roll(텐서, shifts, dims)`:** 정한 축을 따라 값을 shifts칸 밉니다. 밀려 나간 값은 반대쪽 끝으로 들어옵니다(예: `[1, 2, 3, 4, 5]`를 1칸 밀면 `[5, 1, 2, 3, 4]`). 손글씨 그림은 가장자리가 대부분 비어 있어(값 0), 반대쪽으로 들어오는 것도 대부분 0입니다. 바탕이 0이 아닌 그림에서는 이 성격이 맞지 않습니다.
- **뒤집기는 쓰지 않습니다:** 숫자는 좌우로 뒤집으면 2, 7처럼 거울에 비친 다른 모양이 됩니다. 정답이 바뀌는 변환은 쓰면 안 되므로, 이 교안은 숫자 그림에 옮기기만 씁니다. 과제 02의 가위바위보 손 그림은 좌우로 뒤집어도 같은 손 모양이라 뒤집기를 씁니다.

`augment(X_batch)`는 그림마다 위아래와 좌우로 옮길 칸 수를 -2~2 가운데서 따로 골라 옮깁니다.

1. 그림 한 장을 꺼내 위아래 칸 수 `dy`와 좌우 칸 수 `dx`를 고릅니다. 예: `dy = 1`, `dx = -2`면 아래로 1칸, 왼쪽으로 2칸입니다.
2. `torch.roll(그림, shifts=(dy, dx), dims=(1, 2))`로 옮깁니다. 그림 한 장의 모양이 (1, 28, 28)이라 1번 축이 세로, 2번 축이 가로입니다.
3. 모든 그림에 1, 2를 반복합니다. 정답(`y`)은 그대로 둡니다.

In [ ]:
def augment(X_batch):
    """그림마다 위아래, 좌우로 -2~2픽셀을 무작위로 골라 옮긴 사본을 돌려줍니다."""
    shifted = torch.empty_like(X_batch)
    for i in range(len(X_batch)):
        # 옮길 칸 수를 -2, -1, 0, 1, 2 가운데서 고릅니다(randint는 끝값 3을 빼고 고릅니다)
        dy = torch.randint(-2, 3, (1,)).item()
        dx = torch.randint(-2, 3, (1,)).item()

        # 그림 한 장의 모양은 (1, 28, 28)이므로 1번 축이 세로, 2번 축이 가로입니다
        shifted[i] = torch.roll(X_batch[i], shifts=(dy, dx), dims=(1, 2))
    return shifted

In [ ]:
# 학습 그림 한 장을 8번 증강해 원래 그림과 나란히 그립니다
torch.manual_seed(0)
original = X_train[:1]
plt.figure(figsize=(14, 2.2))
plt.subplot(1, 9, 1)
plt.imshow(original[0, 0], cmap="gray_r")
plt.title("원래 그림", fontsize=11)
plt.axis("off")
for k in range(8):
    plt.subplot(1, 9, k + 2)
    plt.imshow(augment(original)[0, 0], cmap="gray_r")
    plt.title(f"증강 {k + 1}", fontsize=11)
    plt.axis("off")
plt.show()

**결과 읽기:** 아홉 장 모두 같은 숫자이고 위치만 1~2픽셀씩 다릅니다. 사람에게는 같은 그림이지만, 모델에게는 픽셀 값이 놓인 자리가 달라 다른 입력입니다.

### 5-2. 증강으로 학습하기

에폭마다 학습 그림 800장 전체를 새로 증강해 `DataLoader`를 다시 만들고, 그 DataLoader로 `train_one_epoch`를 부릅니다. 같은 그림도 에폭마다 다른 위치로 옮겨져 들어갑니다. 모델은 큰 CNN 그대로입니다.

아래 움직이는 그림은 5-1에서 증강한 학습 그림(숫자 9)이 에폭마다 `augment`로 옮겨져 들어가는 모습입니다. 칸 수는 에폭마다 새로 뽑고(그림의 칸 수는 예입니다), 정답은 그대로입니다.

<img src="images/7b_augment_epochs.gif" width="900" alt="움직이는 그림입니다. 제목은 '데이터 증강: 같은 학습 그림을 에폭마다 다르게 옮깁니다'입니다. 왼쪽 '원래 학습 그림'은 MNIST 숫자 9이고 아래에 '정답 9'가 있습니다. 'augment' 화살표 오른쪽 그림이 '에폭 1에 들어가는 그림'부터 '에폭 8에 들어가는 그림'까지 한 장씩 바뀌고, 그 위에 원래 위치(회색 점선 상자), 옮긴 위치(초록 상자), 반대쪽에서 들어온 줄과 칸(주황 띠, 바탕이라 비어 있음)이 표시되며 아래에 '정답 9 (그대로)'가 있습니다. 오른쪽에 'dy = -2, dx = 2'와 '위로 2칸, 오른쪽으로 2칸'처럼 그 에폭의 이동이 적히고, 아래 띠 '에폭마다 뽑은 (dy, dx)'에 이동이 하나씩 쌓입니다.">

In [ ]:
torch.manual_seed(0)
aug_model = make_big_cnn()
aug_optimizer = torch.optim.Adam(aug_model.parameters(), lr=0.001)

aug_rows = []
for epoch in range(1, 31):
    # 에폭마다 학습 그림을 새로 옮겨 DataLoader를 다시 만듭니다. 정답은 그대로입니다
    aug_loader = DataLoader(TensorDataset(augment(X_train), y_train), batch_size=64, shuffle=True, drop_last=True)
    train_loss = train_one_epoch(aug_model, aug_loader, loss_fn, aug_optimizer)

    # 검증 그림은 증강하지 않습니다
    valid_loss, valid_accuracy = evaluate(aug_model, valid_loader, loss_fn)
    aug_rows.append({"에폭": epoch, "학습 손실": train_loss, "검증 손실": valid_loss, "검증 정확도": valid_accuracy})
aug_history = pd.DataFrame(aug_rows)

aug_histories = {"규제 없음": big_history, "증강": aug_history}
plot_histories(aug_histories)
display(summarize(aug_histories).round(4))

**결과 읽기**
- **학습 손실:** 증강하면 30에폭에도 0.0909입니다. 에폭마다 그림이 달라져 800장을 외울 수 없습니다.
- **검증 손실:** 10에폭 무렵까지 빠르게 내려간 뒤, 오르내리면서 천천히 낮아져 30에폭에 0.0783입니다. 규제 없음처럼 다시 커지지 않았고, 지금까지 본 설정 가운데 가장 낮습니다.
- **실무 판단:** 증강은 모델과 옵티마이저를 그대로 두고 데이터만 바꾸므로 다른 규제와 함께 쓰기 쉽습니다. 다만 정답이 바뀌지 않는 변환만 골라야 합니다.

## 6. 조기 종료

규제를 넣어도 학습을 오래 하면 검증 손실이 다시 커질 수 있고, 몇 에폭이 가장 좋은지는 미리 알 수 없습니다. 그래서 학습하면서 검증 손실을 지켜보다가 멈춥니다.

### 6-1. patience와 가장 좋았던 파라미터

**조기 종료(early stopping):** 에폭마다 검증 손실을 계산해, 가장 낮았던 에폭 뒤로 정한 에폭 수(patience) 동안 더 낮아지지 않으면 학습을 멈추는 방법입니다. 가장 좋았던 파라미터는 따로 저장해 두었다가 멈춘 뒤 불러옵니다(이 교안의 함수는 직접 되돌리고, Lightning은 `ckpt_path="best"`로 불러옵니다).

- **patience:** 더 낮아지기를 기다리는 에폭 수입니다. 검증 손실은 에폭마다 조금씩 오르내리므로, 한 번 올랐다고 바로 멈추지 않으려고 기다립니다.
- **체크포인트(checkpoint):** 학습 중 한 시점의 파라미터를 저장해 둔 것입니다. 조기 종료에서는 검증 손실이 가장 낮아질 때마다 저장하고, 멈춘 뒤에는 마지막 파라미터가 아니라 이것을 불러옵니다. 멈춘 시점은 이미 patience 에폭만큼 과적합이 진행된 뒤이기 때문입니다.


<img src="images/75_early_stopping.png" width="1050" alt="제목은 '조기 종료: 검증 손실이 낮아지지 않으면 멈춥니다'입니다. 가로축 '에폭', 세로축 '검증 손실' 그래프에 에폭마다 하나씩 같은 간격의 빨간 점이 있고, 가로축에 에폭 숫자는 적혀 있지 않습니다. 점은 내려가다 가장 좋았던 에폭에 별(초록, '체크포인트 저장' 말풍선)이 있고, 회색 띠 안의 점 10개로 천천히 다시 오릅니다. 띠 위에 'patience 10에폭 동안 더 낮아지지 않음'이 있고, patience 10에폭 뒤 멈춘 에폭에 빨간 점선과 '학습 멈춤'이 있습니다. 별 아래에 '가장 좋았던 에폭', 빨간 점선 아래에 '멈춘 에폭'이 적혀 있습니다. 점선에서 별로 돌아가는 초록 화살표에 '저장한 파라미터 불러오기'가 적혀 있습니다.">


아래 움직이는 그림은 설명용 곡선(실행 결과가 아님)으로 조기 종료를 에폭마다 따라갑니다. 검증 손실이 가장 낮은 값보다 더 낮아지면 체크포인트를 새로 저장하고 기다린 에폭 수를 0부터 다시 셉니다.

<img src="images/7c_early_stopping.gif" width="900" alt="움직이는 그림입니다. 제목은 '조기 종료: 검증 손실이 patience 동안 낮아지지 않으면 멈춥니다'입니다. 왼쪽 그래프(가로축 '에폭', 세로축 '손실', 눈금 없음)에 파란 '학습 손실'과 빨간 '검증 손실'이 에폭마다 점 하나씩 그려지고, 검증 손실이 가장 낮은 점에 초록 별이 있습니다. 오른쪽 'patience 0 / 10' 아래 칸 10개가 더 낮아지지 않은 에폭마다 하나씩 차고, 상자에 '더 낮아지지 않음, 기다린 에폭 수 1 늘리기'가 적힙니다. 앞쪽에서 검증 손실이 두 에폭 오른 뒤 더 낮아지면 별이 옮겨 가고 칸이 모두 비며 '체크포인트 새로 저장, 0부터 다시 셈'이 나옵니다. 별 뒤로 회색 띠 안에서 검증 손실이 조금씩 오르내리며 10칸이 다 차면 'patience 10에 닿음, 학습 멈춤'과 빨간 점선 '학습 멈춤'이 나옵니다. 마지막 장에는 점선에서 별로 가는 초록 화살표와 '저장한 파라미터 불러오기, load_state_dict(best_state)'가 있습니다.">

앞에서 30에폭 학습한 큰 CNN의 곡선에 patience 10을 대어 보면 다음과 같습니다.

1. 10에폭에 검증 손실이 0.1685로 가장 낮아지고, 그때의 파라미터를 저장합니다.
2. 11~20에폭 10번 동안 0.1685보다 낮아지지 않습니다.
3. 20에폭이 끝나면 학습을 멈추고, 저장해 둔 10에폭 파라미터를 불러옵니다.

**저장할 때는 `copy.deepcopy`로 복사합니다.** `model.state_dict()`는 모델이 지금 쓰는 텐서를 가리키는 딕셔너리라, 그대로 변수에 담아 두면 학습이 이어지면서 값이 함께 바뀝니다.

### 6-2. 조기 종료 함수

`fit`에 세 가지를 더한 함수입니다. 증강하는 설정도 같은 함수로 학습하도록, 에폭마다 학습 그림을 새로 옮기는 반복도 넣습니다.

1. 검증 손실이 지금까지 가장 낮으면 파라미터를 `deepcopy`로 저장하고 그 에폭을 기록합니다.
2. 가장 좋았던 에폭에서 patience 에폭이 지나도록 더 낮아지지 않으면 반복을 멈춥니다.
3. 반복이 끝나면 저장한 파라미터를 `load_state_dict`로 모델에 다시 넣습니다.

In [ ]:
import copy


def fit_early_stopping(model, optimizer, max_epochs, patience, use_augment=False):
    """검증 손실이 patience 에폭 동안 낮아지지 않으면 멈추고, 가장 좋았던 파라미터로 되돌린 뒤 (기록 표, 가장 좋았던 에폭)을 돌려줍니다."""
    best_loss = float("inf")  # 무한대에서 시작하면 첫 에폭의 손실이 늘 더 낮습니다
    best_state = None
    best_epoch = 0
    rows = []
    for epoch in range(1, max_epochs + 1):
        # 증강을 쓰면 에폭마다 학습 그림을 새로 옮깁니다
        loader = train_loader
        if use_augment:
            loader = DataLoader(TensorDataset(augment(X_train), y_train), batch_size=64, shuffle=True, drop_last=True)
        train_loss = train_one_epoch(model, loader, loss_fn, optimizer)
        valid_loss, valid_accuracy = evaluate(model, valid_loader, loss_fn)
        rows.append({"에폭": epoch, "학습 손실": train_loss, "검증 손실": valid_loss, "검증 정확도": valid_accuracy})

        # 지금까지 가장 낮은 검증 손실이면 파라미터를 복사해 저장합니다
        if valid_loss < best_loss:
            best_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch

        # 가장 좋았던 에폭에서 patience 에폭이 지나도록 낮아지지 않으면 멈춥니다
        if epoch - best_epoch >= patience:
            break

    # 마지막 파라미터 대신 가장 좋았던 에폭의 파라미터로 되돌립니다
    model.load_state_dict(best_state)
    return pd.DataFrame(rows), best_epoch

In [ ]:
# 큰 CNN을 같은 시드로 만들어 최대 100에폭, patience 10으로 학습합니다
torch.manual_seed(0)
stop_model = make_big_cnn()
stop_optimizer = torch.optim.Adam(stop_model.parameters(), lr=0.001)
stop_history, stop_best_epoch = fit_early_stopping(stop_model, stop_optimizer, max_epochs=100, patience=10)

# 되돌린 모델의 검증 손실은 가장 좋았던 에폭의 검증 손실과 같아야 합니다
restored_loss, restored_accuracy = evaluate(stop_model, valid_loader, loss_fn)
print("멈춘 에폭:", len(stop_history), "| 가장 좋았던 에폭:", stop_best_epoch)
print("되돌린 모델의 검증 손실:", round(restored_loss, 4), "| 검증 정확도:", restored_accuracy)

In [ ]:
# 학습 곡선 위에 가장 좋았던 에폭(초록 점선)과 멈춘 에폭(빨간 점선)을 표시합니다
plt.figure(figsize=(8, 4.5))
plt.plot(stop_history["에폭"], stop_history["학습 손실"], linewidth=2, label="학습 손실")
plt.plot(stop_history["에폭"], stop_history["검증 손실"], linewidth=2, label="검증 손실")
plt.axvline(stop_best_epoch, color="green", linestyle="--", label=f"가장 좋았던 에폭 {stop_best_epoch}")
plt.axvline(len(stop_history), color="red", linestyle="--", label=f"멈춘 에폭 {len(stop_history)}")
plt.xlim(0, 36)  # 오른쪽 빈자리에 범례를 둡니다
plt.ylim(0, 1.0)
plt.xlabel("에폭", fontsize=12)
plt.ylabel("손실", fontsize=12)
plt.title("조기 종료(patience 10)", fontsize=14)
plt.legend(fontsize=11, loc="upper right")
plt.show()

**결과 읽기**
- **멈춘 에폭 20, 가장 좋았던 에폭 10:** 최대 100에폭 가운데 20에폭만 학습하고 멈췄습니다. 가장 좋았던 에폭은 30에폭 학습 곡선에서 찾은 가장 낮은 에폭(10)과 같습니다. 같은 시드라 20에폭까지는 그 곡선과 같습니다.
- **되돌린 모델의 검증 손실 0.1685:** 10에폭의 검증 손실과 같습니다. 마지막(20에폭) 파라미터가 아니라 저장해 둔 파라미터로 되돌아갔습니다.


### 6-3. 조기 종료 모델과 30에폭 모델의 비교

2-3에서 검증 정확도는 비슷한데 검증 손실만 커진 이유를 확인합니다. 되돌린 모델과 30에폭까지 학습한 `big_model`을 검증 그림 200장으로 비교합니다. 그림마다 손실을 따로 구해 틀린 그림과 맞힌 그림으로 나눠 평균합니다.

In [ ]:
# reduction="none"이면 평균하지 않고 그림마다 손실을 하나씩 돌려줍니다
per_image_loss_fn = nn.CrossEntropyLoss(reduction="none")
compare_models = {f"조기 종료({stop_best_epoch}에폭으로 되돌림)": stop_model, "30에폭까지 학습": big_model}

compare_rows = []
for name in compare_models:
    compare_model = compare_models[name]

    # 검증 그림 200장을 한 번에 넣어 그림마다 손실과 예측을 구합니다
    compare_model.eval()
    with torch.no_grad():
        logits = compare_model(X_valid_device)
    image_losses = per_image_loss_fn(logits, y_valid_device)
    predictions = logits.argmax(dim=1)

    # 틀린 그림과 맞힌 그림으로 나눠 손실을 평균합니다
    wrong = predictions != y_valid_device
    correct = predictions == y_valid_device
    compare_rows.append({"모델": name, "틀린 그림 수": wrong.sum().item(),
                         "틀린 그림 평균 손실": image_losses[wrong].mean().item(),
                         "맞힌 그림 평균 손실": image_losses[correct].mean().item()})

compare_table = pd.DataFrame(compare_rows).set_index("모델")
display(compare_table.round(4))

**결과 읽기**
- **틀린 그림 수:** 10장과 12장으로 2장 차이입니다.
- **틀린 그림 평균 손실:** 30에폭 모델이 3.7154로 되돌린 모델(2.3837)보다 큽니다. 틀린 그림의 정답에 더 낮은 확률을 주어, 더 자신 있게 틀립니다.
- **맞힌 그림 평균 손실:** 30에폭 모델이 0.0139로 되돌린 모델(0.0519)보다 작습니다. 맞힌 그림에서 줄어든 손실보다 틀린 그림에서 커진 손실이 커서, 전체 검증 손실이 커졌습니다.

## 7. 규제 비교와 테스트

앞에서는 규제마다 30에폭으로 끊어 비교했으므로, 30에폭 뒤에 검증 손실이 더 내려갈지는 알 수 없었습니다(증강은 30에폭이 가장 낮았습니다). 다섯 설정을 모두 같은 조기 종료(최대 100에폭, patience 10)로 학습해 가장 좋았던 파라미터끼리 비교하고, 하나를 골라 테스트 데이터로 한 번 평가합니다.

### 7-1. 다섯 설정을 조기 종료로 학습하기

**하이퍼파라미터 탐색(hyperparameter search):** 하이퍼파라미터(학습 전에 사람이 정하는 설정, 여기서는 Dropout `p`, weight decay λ, 증강 여부)의 후보 조합을 같은 조건으로 하나씩 학습해, 검증 데이터의 성능(이 교안은 검증 손실)이 가장 좋은 조합을 고르는 일입니다. 테스트 데이터는 탐색에 쓰지 않습니다.

- **격자 탐색(grid search):** 하이퍼파라미터마다 후보를 정해 모든 조합을 학습합니다. 후보가 셋씩인 하이퍼파라미터가 셋이면 3 × 3 × 3 = 27번 학습합니다.
- **무작위 탐색(random search):** 정한 범위 안에서 조합을 무작위로 정한 횟수만큼 뽑아 학습합니다. 하이퍼파라미터가 많아 모든 조합을 학습할 수 없을 때 씁니다([Bergstra & Bengio, 2012](https://jmlr.org/papers/v13/bergstra12a.html)).
- **이 절의 탐색:** Dropout `p`(0, 0.5), λ(0, 1.0), 증강(안 씀, 씀)의 후보 둘씩으로 만든 격자 8조합 가운데 다섯을 골라 학습합니다. 앞 절들의 네 설정과, 둘을 합친 Dropout + 증강 한 조합입니다(모든 조합을 학습하는 격자 탐색은 아닙니다).

다섯 조합의 설정은 다음과 같습니다.

- **규제 없음:** `make_dropout_cnn(0.0)`은 아무것도 0으로 만들지 않으므로 큰 CNN과 같습니다.
- **조합:** 효과가 컸던 Dropout과 증강을 함께 씁니다.
- **옵티마이저:** 모두 AdamW로 만들고 weight decay 설정만 λ = 1.0, 나머지는 0입니다. λ가 0인 AdamW는 Adam과 같은 갱신입니다.
- 다섯 번 학습하므로 CPU에서 1~2분 걸립니다.

In [ ]:
# 설정 이름: (Dropout p, weight_decay, 증강 여부)
settings = {
    "규제 없음": (0.0, 0.0, False),
    "Dropout 0.5": (0.5, 0.0, False),
    "weight decay 1.0": (0.0, 1.0, False),
    "증강": (0.0, 0.0, True),
    "Dropout + 증강": (0.5, 0.0, True),
}

final_models = {}
result_rows = []
for name in settings:
    dropout_p, weight_decay, use_augment = settings[name]

    # 설정마다 같은 시드로 새 모델과 옵티마이저를 만듭니다
    torch.manual_seed(0)
    model = make_dropout_cnn(dropout_p)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=weight_decay)
    history, best_epoch = fit_early_stopping(model, optimizer, max_epochs=100, patience=10, use_augment=use_augment)

    # 모델은 가장 좋았던 에폭의 파라미터로 되돌아와 있습니다
    final_models[name] = model
    best_row = history.loc[best_epoch - 1]  # 표의 행 번호는 0부터, 에폭은 1부터 셉니다

    result_rows.append({"설정": name, "멈춘 에폭": len(history), "가장 좋았던 에폭": best_epoch,
                        "최저 검증 손실": best_row["검증 손실"], "그때 검증 정확도": best_row["검증 정확도"],
                        "에폭 간 변화": epoch_change(history)})

result_table = pd.DataFrame(result_rows)
display(result_table.round(4))

**결과 읽기**
- **규제 없음, Dropout 0.5:** 최저 검증 손실은 앞에서 본 결과와 같습니다(6-2의 10에폭 0.1685, 3-3의 21에폭 0.1179). AdamW의 weight decay를 0으로 두면 Adam과 같은 갱신이기 때문입니다. 다만 규제 없음은 20에폭에 멈춰 15~20에폭만으로 에폭 간 변화를 계산하므로, 30에폭까지 계산한 2-3의 0.0071과 달리 0.0093입니다.
- **weight decay 1.0:** 4-2와 같은 13에폭에 0.1752이고 23에폭에 멈춥니다. 규제 없음과의 차이 0.007은 에폭 간 변화(0.0093, 0.0132)보다 작아 효과를 단정할 수 없습니다.
- **증강:** 30에폭(0.0783)이 가장 낮았고, 그 뒤 10에폭 동안 더 낮아지지 않아 40에폭에 멈춥니다. 30에폭으로 끊은 증강 학습이 이미 가장 좋은 때였습니다.
- **Dropout + 증강:** 43에폭에 0.0827로 증강과 비슷하고, 53에폭에 멈춥니다. 검증 손실이 늦게까지 내려가 가장 오래 학습합니다.
- **에폭 간 변화:** 설정마다 0.0093~0.0157입니다(정의는 2-3).

### 7-2. 고른 설정으로 테스트 한 번

검증 손실이 가장 낮은 설정 하나를 고르고, 그 모델만 테스트 데이터 2,000장으로 평가합니다. 이 숫자가 보고할 성능입니다.

- **증강과 Dropout + 증강:** 최저 검증 손실의 차이 0.0044(0.0783, 0.0827)가 두 설정의 에폭 간 변화(0.0157, 0.0154)보다 작아 두 설정을 가를 수 없습니다.
- **고르는 규칙:** 그래도 하나를 골라야 하므로, 정한 규칙대로 최저 검증 손실이 가장 낮은 설정을 고릅니다.

In [ ]:
# 최저 검증 손실이 가장 낮은 설정을 고릅니다
chosen_name = result_table.loc[result_table["최저 검증 손실"].idxmin(), "설정"]
chosen_model = final_models[chosen_name]
print("고른 설정:", chosen_name)

# 고른 모델만 테스트 데이터로 한 번 평가합니다
test_loss, test_accuracy = evaluate(chosen_model, test_loader, loss_fn)
print("테스트 손실:", round(test_loss, 4), "| 테스트 정확도:", test_accuracy)

**결과 읽기:** 규칙대로 `증강`을 골랐고, 테스트 정확도는 0.9605입니다. 학습 그림 800장만으로 학습했지만 처음 보는 그림 2,000장에서 이만큼 맞힙니다. 실행 환경에 따라 0.01 안팎 달라질 수 있습니다.

**확인 퀴즈**: 다섯 설정을 모두 테스트 데이터로 평가해 정확도가 가장 높은 설정을 보고해도 될까요?

<details><summary>정답 보기</summary>

안 됩니다.

- **테스트 데이터로 고르면:** 테스트 정확도가 가장 높은 설정을 고른 것이 되어, 그 정확도는 처음 보는 데이터의 성능보다 좋게 나옵니다. 검증 데이터가 하던 일을 테스트 데이터에 맡긴 셈이고, 더는 처음 보는 데이터가 아닙니다.
- **바꾸고 싶으면:** 검증 데이터로 다시 고르고, 고르는 데 쓰지 않은 새 테스트 데이터로 평가합니다.

</details>

## 8. PyTorch Lightning

지금까지는 학습 반복, 모드 바꾸기, 기록, 조기 종료, 체크포인트를 모두 손으로 썼습니다. 이 부분은 모델이 바뀌어도 거의 같으므로, 정해진 틀로 대신 돌려 주는 라이브러리를 씁니다.

### 8-1. Lightning이 대신하는 것

**PyTorch Lightning:** PyTorch 학습 코드를 두 부분으로 나눠 쓰게 하는 라이브러리입니다([Lightning 문서](https://lightning.ai/docs/pytorch/stable/)).

- **`LightningModule`:** 모델, 미니배치 하나의 손실 계산, 옵티마이저처럼 모델마다 다른 부분을 내가 씁니다.
- **`Trainer`:** 에폭 반복, `zero_grad`·`backward`·`step`, 학습·평가 모드 바꾸기, 계산 장치로 옮기기처럼 늘 같은 부분을 Lightning이 합니다.
- **콜백(callback):** Trainer가 학습 중 정해진 시점마다 불러 주는 부품입니다. 조기 종료(`EarlyStopping`)와 체크포인트 저장(`ModelCheckpoint`)은 에폭마다 검증이 끝난 뒤 `val_loss`를 보고 작동하는 콜백입니다.


<img src="images/76_lightning_map.png" width="1050" alt="제목은 'Lightning: 내가 쓰는 것과 Trainer가 하는 것'입니다. 왼쪽 큰 상자 'LightningModule (내가 씀)' 안에 위에서 아래로 '__init__: 모델 make_dropout_cnn(0.5)', 'training_step: 미니배치 손실', 'validation_step: 검증 손실·정확도', 'configure_optimizers: Adam'이 있습니다. 오른쪽 큰 상자 'Trainer (Lightning이 함)' 안에 '에폭 반복 max_epochs', 'zero_grad → backward → step', 'model.train() / model.eval()', '.to(장치)'가 있습니다. 오른쪽 상자 아래에 작은 상자 세 개 'EarlyStopping: patience 세기', 'ModelCheckpoint: 가장 좋은 파라미터 파일', 'CSVLogger: 기록 표 metrics.csv'가 매달려 있습니다. 두 큰 상자 사이 화살표에 'trainer.fit(모델, 학습 DataLoader, 검증 DataLoader)'가 적혀 있습니다.">


지금까지 손으로 쓴 코드와 짝을 지으면 다음과 같습니다.

| 손으로 쓴 코드 | Lightning |
|---|---|
| `make_dropout_cnn(0.5)`로 모델 만들기 | `__init__` 안의 `self.model = make_dropout_cnn(0.5)` |
| `train_one_epoch` 안의 손실 계산 | `training_step`(미니배치 하나의 손실을 돌려줌) |
| `zero_grad`, `backward`, `step`, `model.train()` | Trainer가 대신 함 |
| `evaluate(model, valid_loader, loss_fn)` | `validation_step`(`model.eval()`과 `torch.no_grad()`는 Trainer가 함) |
| `torch.optim.Adam(...)` | `configure_optimizers` |
| `for epoch in range(1, 101)` | `Trainer(max_epochs=100)` |
| 가장 좋았던 에폭에서 patience 세기 | `EarlyStopping(monitor="val_loss", patience=10)` |
| `copy.deepcopy(model.state_dict())` | `ModelCheckpoint(monitor="val_loss")`(파일로 저장) |
| 가장 좋은 파라미터로 되돌리기: `model.load_state_dict(best_state)` | 자동으로 하지 않음, `ckpt_path="best"`로 불러옴 |
| `rows.append(...)`로 기록 표 만들기 | `self.log(...)` + `CSVLogger` |
| `evaluate(model, test_loader, loss_fn)` | `test_step` + `trainer.test(...)` |

### 8-2. 클래스 상속

파이썬 기초에서 배운 클래스 상속을 씁니다. `class LitCNN(L.LightningModule)`로 물려받고 `__init__` 첫 줄에서 `super().__init__()`을 부르면, Trainer가 정해진 이름의 메서드(`training_step` 등)를 알맞은 때에 부릅니다.

### 8-3. LightningModule 만들기

Dropout CNN(`make_dropout_cnn`)을 그대로 감쌉니다. `self.log("이름", 값)`으로 남긴 값은 검증 단계(`validation_step`)에서는 에폭마다 평균되어 기록되고, 학습 단계(`training_step`)에서는 `on_epoch=True`를 주어야 에폭 평균으로 기록됩니다. 조기 종료와 체크포인트는 그 이름(`val_loss`)을 보고 판단합니다.

In [ ]:
import logging
import warnings

import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

# Lightning이 실행마다 띄우는 안내 문구(장치 정보, 다른 서비스 소개, 성능 권고)를 숨기고 결과만 봅니다
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", module="lightning")

In [ ]:
class LitCNN(L.LightningModule):  # LightningModule의 기능을 물려받는 새 클래스
    """make_dropout_cnn 모델을 Lightning의 학습 틀에 맞춰 감싼 클래스입니다."""

    def __init__(self, dropout_p, lr):  # 객체를 만들 때 한 번 실행됩니다. self는 만들어지는 객체 자신입니다
        super().__init__()                          # 물려받은 LightningModule의 준비를 먼저 합니다
        self.save_hyperparameters()                 # 인자(dropout_p, lr)를 체크포인트 파일에 함께 저장합니다
        self.model = make_dropout_cnn(dropout_p)    # Dropout CNN을 객체에 붙여 둡니다
        self.loss_fn = nn.CrossEntropyLoss()        # 손실 함수도 붙여 둡니다
        self.lr = lr                                # configure_optimizers에서 쓸 학습률

    def forward(self, X_batch):  # lit_model(X_batch)처럼 부르면 실행됩니다
        return self.model(X_batch)

    def training_step(self, batch, batch_idx):  # 학습 미니배치 하나마다 Trainer가 부릅니다
        X_batch, y_batch = batch
        loss = self.loss_fn(self(X_batch), y_batch)  # self(X_batch)는 위의 forward를 부릅니다
        self.log("train_loss", loss, on_step=False, on_epoch=True)  # 스텝마다가 아니라 에폭 평균으로 기록
        return loss  # 돌려준 손실로 Trainer가 zero_grad, backward, step을 합니다

    def validation_step(self, batch, batch_idx):  # 검증 미니배치 하나마다 Trainer가 부릅니다(평가 모드, 기울기 없이)
        X_batch, y_batch = batch
        logits = self(X_batch)
        self.log("val_loss", self.loss_fn(logits, y_batch))  # 조기 종료와 체크포인트가 이 이름을 봅니다
        self.log("val_acc", (logits.argmax(dim=1) == y_batch).float().mean())

    def test_step(self, batch, batch_idx):  # trainer.test가 테스트 미니배치마다 부릅니다(함께 따라하기 마지막에서 씁니다)
        X_batch, y_batch = batch
        logits = self(X_batch)
        self.log("test_loss", self.loss_fn(logits, y_batch))
        self.log("test_acc", (logits.argmax(dim=1) == y_batch).float().mean())

    def configure_optimizers(self):  # 학습을 시작할 때 Trainer가 한 번 부릅니다
        return torch.optim.Adam(self.parameters(), lr=self.lr)

### 8-4. Trainer로 학습하기

손으로 쓴 조기 종료와 같은 조건(최대 100에폭, patience 10)으로 학습합니다.

- **`EarlyStopping(monitor="val_loss", patience=10)`:** `val_loss`가 10번의 검증(이 교안은 에폭마다 검증하므로 10에폭) 동안 낮아지지 않으면 멈춥니다. 멈추기만 하고 파라미터를 되돌리지는 않습니다(8-6).
- **`ModelCheckpoint(monitor="val_loss")`:** `val_loss`가 가장 낮을 때의 파라미터를 파일(`.ckpt`)로 저장합니다. `deepcopy` 대신 파일에 남기므로 노트북을 닫아도 남습니다.
- **`CSVLogger`:** `self.log`로 남긴 값을 `output/` 아래 `metrics.csv`에 에폭마다 적습니다. 다시 실행하면 `version_0`, `version_1`처럼 새 폴더에 적습니다.
- **`accelerator`:** 준비 셀에서 고른 계산 장치(cuda 또는 cpu)입니다. `enable_progress_bar=False`와 `enable_model_summary=False`는 진행 막대와 층 요약표를 숨깁니다.
- **`deterministic="warn"`:** 같은 시드면 같은 결과가 나오게 계산합니다. GPU에서 그렇게 계산할 방법이 없는 연산을 만나면 멈추지 않고 경고만 냅니다.
- **`L.seed_everything(0, verbose=False)`:** `torch.manual_seed(0)`처럼 시드를 고정합니다. `verbose=False`는 고정했다는 안내 문구를 끕니다.

In [ ]:
# 시드를 고정하고 모델 객체를 만듭니다
L.seed_everything(0, verbose=False)
lit_model = LitCNN(dropout_p=0.5, lr=0.001)

# 조기 종료, 체크포인트, 기록을 맡을 부품
early_stopping = EarlyStopping(monitor="val_loss", patience=10)
checkpoint = ModelCheckpoint(monitor="val_loss")
logger = CSVLogger(save_dir=material_dir / "output", name="lesson02_mnist")

# 최대 100에폭 동안 학습합니다. 진행 막대와 층 요약표는 숨깁니다
trainer = L.Trainer(max_epochs=100, callbacks=[early_stopping, checkpoint], logger=logger,
                    accelerator=device.type, enable_progress_bar=False, enable_model_summary=False, deterministic="warn")
trainer.fit(lit_model, train_loader, valid_loader)

# 체크포인트 파일 이름의 epoch는 0부터 셉니다(epoch=20이면 21번째 에폭)
print("학습한 에폭 수:", trainer.current_epoch)
print("가장 낮은 val_loss:", round(checkpoint.best_model_score.item(), 4))
print("체크포인트 파일:", Path(checkpoint.best_model_path).name)

**결과 읽기**
- **멈춘 에폭과 가장 낮은 `val_loss`:** 31에폭에서 멈췄고, 가장 낮은 `val_loss`는 0.1179(21번째 에폭, 파일 이름은 `epoch=20`)입니다.
- **손으로 쓴 반복과 비교:** 다섯 설정 비교 표의 `Dropout 0.5`(31에폭에 멈춤, 21에폭 0.1179)와 같습니다. 같은 시드로 같은 모델, 미니배치 순서, 조기 종료 조건을 썼으므로 같은 계산을 했습니다.
- **`lit_model`의 파라미터:** `fit`이 끝난 `lit_model`은 마지막 에폭의 파라미터입니다. 손으로 쓴 `fit_early_stopping`과 달리 가장 좋았던 파라미터로 저절로 되돌리지 않습니다.

이 교안의 Trainer에는 넣지 않았지만 실무 학습에서 자주 켜는 인자가 하나 더 있습니다.

**기울기 자르기(gradient clipping):** 기울기가 기준값보다 커지지 않게 줄이는 방법입니다. 가장 많이 쓰는 노름 기준 자르기는 모든 파라미터의 기울기를 한 벡터로 이어 붙인 크기(L2 노름)가 기준값보다 크면, 기울기 전체에 같은 비율을 곱해 크기를 기준값으로 줄입니다. 방향은 그대로입니다. 목적은 교안 01 5-1의 기울기 폭주처럼 기울기가 갑자기 커질 때 한 번의 갱신으로 파라미터가 크게 튀지 않게 하는 것입니다.

$\displaystyle g \leftarrow g \times \min\left(1,\ \frac{\text{기준값}}{\lVert g \rVert}\right)$


<img src="images/79_gradient_clipping.png" width="1050" alt="제목은 '기울기 자르기: 방향은 그대로, 크기만 기준값으로 줄입니다'입니다. 왼쪽 좌표평면(가로축 '파라미터 1의 기울기', 세로축 '파라미터 2의 기울기', 눈금 0~4)에 반지름 1인 점선 원 '기준값 1'이 있고, 원점에서 (3, 4)까지 보라 화살표 '자르기 전 (3, 4) 크기 5'와, 같은 방향으로 원 위의 (0.6, 0.8)까지 초록 화살표 '자른 뒤 (0.6, 0.8) 크기 1'이 있습니다. 오른쪽 카드 두 장은 '1 기울기 전체의 크기: √(3² + 4²) = 5'와 '2 5 > 1 이면 1 / 5 = 0.2 곱하기: (3, 4) × 0.2 = (0.6, 0.8)'입니다. 아래 띠 두 줄에 'loss.backward() -> clip_grad_norm_(..., max_norm=1.0) -> optimizer.step()'과 '기울기 폭주로 한 번에 크게 튀는 갱신을 막습니다'가 적혀 있습니다.">


- **PyTorch:** `loss.backward()`와 `optimizer.step()` 사이에 `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)`을 넣습니다. 돌려주는 값은 자르기 전 크기입니다.
- **Lightning:** `L.Trainer(gradient_clip_val=1.0)`으로 주면 Trainer가 같은 자리에서 자릅니다(`gradient_clip_algorithm`을 적지 않으면 `clip_grad_norm_`과 같은 노름 기준). 기본값은 `None`(자르지 않음)이고, 이 교안의 학습은 손실이 발산하지 않아 쓰지 않았습니다.
- **그림의 계산:** 기울기 (3, 4)의 크기는 $\sqrt{3^2 + 4^2}$ = 5로 기준값 1보다 크므로 1.0 / 5 = 0.2를 곱해 (0.6, 0.8), 크기 1.0이 됩니다. 두 값의 비율 3 : 4가 그대로라 갱신 방향은 같고 크기만 줄었습니다.

### 8-5. 기록으로 학습 곡선 그리기

`metrics.csv`에는 학습 손실과 검증 값이 서로 다른 행에 적히므로, 에폭마다 한 행으로 묶어 읽는 함수를 둡니다. 함께 따라하기에서도 씁니다.

In [ ]:
def read_metrics(trainer):
    """CSVLogger가 저장한 metrics.csv를 읽어, 에폭마다 한 행(train_loss, val_loss, val_acc)인 표로 돌려줍니다."""
    metrics = pd.read_csv(Path(trainer.logger.log_dir) / "metrics.csv")

    # 같은 에폭의 학습 행과 검증 행을 묶습니다. 빈 칸(NaN)은 평균에서 빠집니다
    epoch_table = metrics.groupby("epoch")[["train_loss", "val_loss", "val_acc"]].mean().reset_index()
    epoch_table["epoch"] = epoch_table["epoch"] + 1  # Lightning은 에폭을 0부터 셉니다
    return epoch_table


lit_metrics = read_metrics(trainer)
display(lit_metrics.tail(3).set_index("epoch").round(4))

In [ ]:
# Lightning 기록으로 그린 학습 곡선. 점선은 val_loss가 가장 낮은 에폭입니다
lit_best_epoch = lit_metrics.loc[lit_metrics["val_loss"].idxmin(), "epoch"]
plt.figure(figsize=(8, 4.5))
plt.plot(lit_metrics["epoch"], lit_metrics["train_loss"], linewidth=2, label="train_loss")
plt.plot(lit_metrics["epoch"], lit_metrics["val_loss"], linewidth=2, label="val_loss")
plt.axvline(lit_best_epoch, color="gray", linestyle="--")
plt.ylim(0, 1.0)
plt.xlabel("에폭", fontsize=12)
plt.ylabel("손실", fontsize=12)
plt.title("Lightning 기록(CSVLogger)으로 그린 학습 곡선", fontsize=14)
plt.legend(fontsize=11)
plt.show()

**결과 읽기:** 손으로 쓴 `fit`의 기록 표와 같은 모양의 곡선이 `self.log` 몇 줄로 남았습니다. `train_loss`는 Dropout이 켜진 채로 계산한 값이라 15에폭 무렵까지 `val_loss`보다 높습니다.

### 8-6. 가장 좋은 체크포인트 불러오기

`trainer.validate(..., ckpt_path="best")`는 `ModelCheckpoint`가 저장한 가장 좋은 파일을 모델에 불러온 뒤, 검증 데이터로 `validation_step`을 돌려 `self.log`로 남긴 값을 돌려줍니다. `Dropout 0.5`는 검증 손실로 고른 설정이 아니므로 테스트 데이터는 쓰지 않고, 가장 좋은 파라미터가 불러와졌는지만 검증 손실로 확인합니다.

In [ ]:
# 가장 좋은 체크포인트를 불러와 검증 데이터로 다시 평가합니다. verbose=False는 결과 표 출력을 끕니다
valid_results = trainer.validate(lit_model, dataloaders=valid_loader, ckpt_path="best", verbose=False)

# 돌려준 값은 val_loss, val_acc를 담은 딕셔너리 하나가 든 리스트입니다
print("불러온 모델의 val_loss:", round(valid_results[0]["val_loss"], 4), "| val_acc:", round(valid_results[0]["val_acc"], 4))
print("체크포인트가 기록한 가장 낮은 val_loss:", round(checkpoint.best_model_score.item(), 4))

체크포인트 파일에는 파라미터와 함께 `save_hyperparameters()`로 저장한 인자가 들어 있으므로, 새 세션에서도 `LitCNN.load_from_checkpoint(파일 경로)` 한 줄로 같은 모델을 다시 만들어 불러올 수 있습니다.

In [ ]:
# 체크포인트 파일에서 모델을 새로 만들어 불러오고, 같은 검증 데이터로 평가합니다
loaded_lit_model = LitCNN.load_from_checkpoint(checkpoint.best_model_path)
loaded_loss, loaded_accuracy = evaluate(loaded_lit_model.model, valid_loader, loss_fn)
print("파일에서 불러온 모델의 검증 손실:", round(loaded_loss, 4), "| 검증 정확도:", loaded_accuracy)

**결과 읽기**
- **불러온 모델의 `val_loss` 0.1179:** 체크포인트가 기록한 가장 낮은 값과 같고, 기록 표의 마지막 에폭 값(0.1338)과는 다릅니다. 마지막 에폭이 아니라 가장 좋았던 에폭의 파라미터가 불러와졌습니다.
- **파일에서 불러온 모델의 검증 손실 0.1179:** 같은 값입니다. 모델을 만드는 인자를 따로 적지 않아도 파일에 저장된 `dropout_p`와 `lr`로 같은 구조가 만들어졌습니다.
- **테스트는 고른 모델에 한 번만:** `test_step`과 `trainer.test`는 함께 따라하기 마지막에서 한 번 씁니다.
- **증강을 더하려면:** `training_step` 첫 줄에서 `X_batch = augment(X_batch)`로 미니배치를 바꾸면 됩니다.

### 8-7. 옵티마이저 상태와 이어 학습

체크포인트 파일에는 파라미터 말고도 학습을 멈춘 지점부터 이어 가는 데 필요한 값이 들어 있습니다.

**옵티마이저 상태(optimizer state):** 옵티마이저가 갱신과 갱신 사이에 기억해 두는 값입니다. Adam과 AdamW는 파라미터마다 두 이동평균을 두고, 이번 스텝의 기울기 $g$로 아래처럼 갱신합니다(기본값 `betas=(0.9, 0.999)`, $\eta$는 학습률).

$\displaystyle m \leftarrow 0.9\, m + 0.1\, g \qquad v \leftarrow 0.999\, v + 0.001\, g^2 \qquad w \leftarrow w - \eta \, \frac{\hat{m}}{\sqrt{\hat{v}} + \epsilon}$

- **1차 모멘트 $m$(`exp_avg`):** 기울기의 이동평균입니다. 갱신 방향을 정합니다.
- **2차 모멘트 $v$(`exp_avg_sq`):** 기울기 제곱의 이동평균입니다. 그 파라미터의 기울기가 대략 얼마나 큰지 기억해, $m$을 $\sqrt{v}$로 나눠 파라미터마다 갱신 크기를 맞춥니다(지난 시간 옵티마이저 표의 Adam).
- **$\hat{m}$, $\hat{v}$:** $m$, $v$가 0에서 시작하므로, $t$번째 스텝에서 $m$을 $1 - 0.9^t$로, $v$를 $1 - 0.999^t$로 나눠 키운 값입니다.

기울기가 0.2인 파라미터의 첫 갱신($t$ = 1, 학습률 0.001)을 따라가 봅니다.

1. $m$ = 0.1 × 0.2 = 0.02, $v$ = 0.001 × 0.2² = 0.00004
2. $\hat{m}$ = 0.02 / 0.1 = 0.2, $\hat{v}$ = 0.00004 / 0.001 = 0.04
3. 갱신 크기 = 0.001 × 0.2 / $\sqrt{0.04}$ = 0.001입니다. 기울기가 20이어도 첫 갱신은 같은 0.001이고, 스텝이 쌓이면서 $m$, $v$가 그 파라미터의 기울기 기록을 담게 됩니다.

**이어 학습(resume training):** 멈춘 학습을 멈춘 지점부터 다시 하는 것입니다. 파라미터만 불러오면 $m$, $v$가 0으로 돌아가 모든 파라미터가 위의 첫 갱신부터 다시 시작하므로, 파라미터와 함께 옵티마이저 상태와 에폭 수를 저장했다가 되살립니다. 8-4의 체크포인트 파일에 무엇이 들어 있는지 열어 봅니다.

In [ ]:
# 체크포인트 파일(.ckpt)을 딕셔너리로 읽습니다. map_location="cpu"는 GPU에서 저장한 파일도 CPU로 읽게 합니다
ckpt = torch.load(checkpoint.best_model_path, map_location="cpu")
print("저장된 항목:", list(ckpt.keys()))
print("저장한 에폭(0부터):", ckpt["epoch"], "| 갱신 횟수:", ckpt["global_step"])

# 옵티마이저 상태: 파라미터 텐서마다 step, exp_avg(m), exp_avg_sq(v)가 있습니다. 0번이 첫 합성곱 층의 가중치입니다
first_state = ckpt["optimizer_states"][0]["state"][0]
print("상태 항목:", list(first_state.keys()), "| step:", int(first_state["step"].item()))
print("가중치 모양:", tuple(ckpt["state_dict"]["model.0.weight"].shape), "| exp_avg:", tuple(first_state["exp_avg"].shape), "| exp_avg_sq:", tuple(first_state["exp_avg_sq"].shape))

**결과 읽기**
- **저장된 항목:** 파라미터(`state_dict`) 말고도 옵티마이저 상태(`optimizer_states`), 스케줄러 상태(`lr_schedulers`, 스케줄러가 없어 빈 목록), 에폭(`epoch`)과 갱신 횟수(`global_step`), 조기 종료가 센 값 같은 콜백 상태(`callbacks`)가 들어 있습니다.
- **에폭 20, 갱신 252번:** 0부터 센 20번째, 곧 21번째 에폭이 끝난 때의 파일이고, 12스텝 × 21에폭 = 252입니다. 옵티마이저 상태의 `step`도 252입니다.
- **`exp_avg`, `exp_avg_sq`:** 첫 가중치와 같은 (32, 1, 3, 3) 모양입니다. 파라미터 하나마다 m, v가 하나씩이라 상태의 숫자는 파라미터 수의 두 배이고, Adam으로 학습하면 파라미터 크기의 두 배만큼 메모리가 더 듭니다.
- **이어 학습하는 방법:** `trainer.fit(lit_model, train_loader, valid_loader, ckpt_path=파일 경로)`로 부르면 Lightning이 이 파일에서 파라미터, 옵티마이저 상태, 에폭 수를 되살려 다음 에폭부터 학습합니다. 손으로 쓴 학습이면 `model.state_dict()`와 함께 `optimizer.state_dict()`와 에폭 수를 `torch.save`로 저장하고, 이어 할 때 `optimizer.load_state_dict(...)`로 되살립니다.

### 8-8. Hugging Face Trainer의 설정 이름

LLM 파인튜닝 단원에서는 Hugging Face `Trainer`로 학습하고, 설정을 `TrainingArguments`에 이름으로 적습니다. 이 교안에서 손으로 쓰고 Lightning으로 바꾼 기능이 각각 어느 이름인지 모으면 다음과 같습니다(에폭 수, 학습률, weight decay, 스케줄러, warmup, 배치 크기의 이름과 기본값은 딥러닝 학습 원리 단원 교안 02 7-4의 표).

| 기능 | 이 교안의 코드 | Lightning | Hugging Face `Trainer`(`TrainingArguments` 기본값) |
|---|---|---|---|
| 검증 주기 | 에폭마다 `evaluate` | 에폭마다 `validation_step` | `eval_strategy="epoch"` (`"no"`: 검증하지 않음) |
| 체크포인트 저장 | `copy.deepcopy(model.state_dict())` | `ModelCheckpoint(monitor="val_loss")` | `save_strategy="epoch"` (`"steps"`) |
| 가장 좋은 체크포인트 다시 불러오기 | `model.load_state_dict(best_state)` | `ckpt_path="best"` | `load_best_model_at_end=True` (False), `metric_for_best_model="eval_loss"` (None) |
| 조기 종료 | `patience=10` | `EarlyStopping(monitor="val_loss", patience=10)` | `EarlyStoppingCallback(early_stopping_patience=10)` (1) |
| 기울기 자르기 | `clip_grad_norm_(..., max_norm=1.0)` | `Trainer(gradient_clip_val=1.0)` | `max_grad_norm` (1.0) |
| 이어 학습 | `optimizer.load_state_dict(...)` | `trainer.fit(..., ckpt_path=파일 경로)` | `trainer.train(resume_from_checkpoint=체크포인트 폴더)` |

- **기본값이 다른 것:** 기울기 자르기는 Lightning이 기본으로 하지 않고(`gradient_clip_val=None`), `TrainingArguments`는 기본으로 합니다(`max_grad_norm=1.0`).
- **가장 좋은 체크포인트:** `load_best_model_at_end=True`이면 `save_strategy`와 `eval_strategy`를 같게(둘 다 `"epoch"`) 맞춰야 하고, 학습이 끝날 때 `metric_for_best_model`이 가장 좋았던 체크포인트를 불러옵니다. `metric_for_best_model`을 적지 않으면 검증 손실이 기준입니다. Lightning의 `fit`이 마지막 파라미터로 끝나는 것(8-4)과 다른 점입니다.
- **조기 종료:** `EarlyStoppingCallback`은 `eval_strategy`가 `"no"`가 아니고 `metric_for_best_model`이 정해져 있어야 작동합니다. `load_best_model_at_end=True`가 아니면 경고를 내고, 멈춘 뒤 가장 좋은 체크포인트를 불러오지 않습니다.
- 기본값은 transformers 5.13, lightning 2.6.6, torch 2.11 기준입니다([TrainingArguments 문서](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments)).

## 9. 🖐️ 함께 따라하기: 옷 사진 분류의 과적합 줄이기

옷 사진(Fashion-MNIST)도 학습 그림이 적으면 과적합이 생기는지 진단하고, 기울기 자르기와 이어 학습, Dropout `p` 탐색을 Lightning으로 직접 적용해 확인합니다.

**본실습과 다른 점**
- **데이터:** 28 × 28 흑백 옷 사진 10종류인 [Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist)(Xiao, Rasul & Vollgraf, 2017, MIT 라이선스) 표본입니다(`data/따라하기/fashion_small.pt`). 교안 01 따라하기에서 본 대로 티셔츠와 셔츠처럼 비슷한 종류가 있어 손글씨보다 어렵습니다.
- **나누기:** 학습 400장, 검증 200장으로 본실습보다 학습 그림을 더 줄여 과적합을 빨리 일으킵니다.
- **학습:** `fit` 대신 처음부터 Lightning으로 학습합니다. 규제 없는 모델도 `LitCNN(dropout_p=0.0, ...)`으로 만듭니다.
- **새로 켜는 것:** 본실습 Trainer에는 넣지 않은 기울기 자르기(8-4)와 이어 학습(8-7)을 직접 켜고, Dropout `p`를 검증 손실로 고르는 작은 탐색(7-1)을 합니다.

**실습 진행 방법:** 작성 셀의 `...`만 주석에 맞춰 채웁니다. 완성 코드는 `정답/교안_02_정답.ipynb`의 같은 절에 있습니다.

**설정**
- 학습 400장, 검증 200장(순서를 섞는 시드 0), 테스트 2,000장
- 모델 `LitCNN`, Adam 학습률 0.001, 배치 크기 64
- 규제 없음: 30에폭
- 이어 학습: Dropout 0.5, 5에폭 학습해 저장한 뒤 10에폭까지 이어 학습, `gradient_clip_val=1.0`
- Dropout `p` 탐색: 후보 0.25, 0.5, 0.75, 최대 100에폭, patience 10, `gradient_clip_val=1.0`
- 기울기 자르기 기준값 1.0은 Hugging Face `TrainingArguments`의 기본값(`max_grad_norm=1.0`, 8-8)과 같습니다.

### 9-1. 데이터 나누기와 DataLoader

In [ ]:
# 옷 사진을 읽고 픽셀을 0~1, (그림 수, 1, 28, 28) 모양으로 바꿉니다
fashion = torch.load(data_dir / "따라하기" / "fashion_small.pt")
fashion_names = fashion["class_names"]
fashion_images = (fashion["train_images"].float() / 255).unsqueeze(1)
fashion_labels = fashion["train_labels"]
fashion_order = torch.randperm(len(fashion_labels), generator=torch.Generator().manual_seed(0))
print("종류:", fashion_names)

# 🖐️ 함께 따라하기: 섞은 순서의 앞 400장은 학습, 다음 200장은 검증으로 나누고, 테스트 2,000장도 준비하세요
X_fashion_train = ...
y_fashion_train = ...
X_fashion_valid = ...
y_fashion_valid = ...
X_fashion_test = ...
y_fashion_test = ...

# 🖐️ 세 DataLoader를 만드세요(배치 크기 64, 학습용만 섞고 drop_last=True로 마지막 미니배치를 버립니다)
fashion_train_loader = ...
fashion_valid_loader = ...
fashion_test_loader = ...
print("학습:", X_fashion_train.shape, "| 검증:", X_fashion_valid.shape, "| 테스트:", X_fashion_test.shape)

### 9-2. 규제 없이 학습해 과적합 진단하기

`LitCNN(dropout_p=0.0, lr=0.001)`을 콜백 없이 30에폭 학습하고, 기록으로 학습 곡선을 그립니다.

In [ ]:
L.seed_everything(0, verbose=False)

# 🖐️ 함께 따라하기: 규제 없는 LitCNN과 CSVLogger(name="lesson02_fashion_no_reg"), 30에폭 Trainer를 만들어 학습하세요
fashion_plain_model = ...
fashion_plain_trainer = ...
...

# 🖐️ 기록을 에폭마다 한 행인 표로 읽으세요
fashion_plain_metrics = ...
display(fashion_plain_metrics.loc[[fashion_plain_metrics["val_loss"].idxmin(), len(fashion_plain_metrics) - 1]].set_index("epoch").round(4))

In [ ]:
# 규제 없는 모델의 학습 곡선
plt.figure(figsize=(8, 4.5))
plt.plot(fashion_plain_metrics["epoch"], fashion_plain_metrics["train_loss"], linewidth=2, label="train_loss")
plt.plot(fashion_plain_metrics["epoch"], fashion_plain_metrics["val_loss"], linewidth=2, label="val_loss")
plt.ylim(0, 1.5)
plt.xlabel("에폭", fontsize=12)
plt.ylabel("손실", fontsize=12)
plt.title("옷 사진, 규제 없음", fontsize=14)
plt.legend(fontsize=11)
plt.show()

**결과 읽기**
- **`val_loss` 18에폭 0.5297, 30에폭 0.6259:** 가장 낮았던 에폭 뒤로 다시 커졌습니다.
- **`train_loss` 30에폭 0.2448:** 그동안 학습 손실은 계속 줄었으므로 과적합입니다. 손글씨보다 손실이 전체적으로 높지만 진단하는 방법은 같습니다.

### 9-3. 기울기 자르기와 이어 학습

학습을 5에폭에서 끊고, 저장한 체크포인트 파일에서 10에폭까지 이어 학습합니다.

1. `L.Trainer(..., gradient_clip_val=1.0)`으로 기울기 자르기를 켜고 5에폭 학습합니다. `ModelCheckpoint()`를 `monitor` 없이 만들면 에폭이 끝날 때마다 마지막 에폭의 파라미터를 저장합니다.
2. 새 `LitCNN`과 `max_epochs=10`인 새 Trainer를 만들고 `fit(..., ckpt_path=파일 경로)`로 부릅니다. Trainer가 파일에서 파라미터, 옵티마이저 상태, 에폭 수를 되살려 6에폭부터 학습합니다.

In [ ]:
L.seed_everything(0, verbose=False)

# 🖐️ 함께 따라하기: Dropout 0.5인 LitCNN, monitor 없는 ModelCheckpoint, CSVLogger(name="lesson02_fashion_resume")를 붙이고 gradient_clip_val=1.0인 5에폭 Trainer로 학습하세요
resume_model = ...
resume_checkpoint = ...
first_trainer = ...
...
print("저장한 파일:", Path(resume_checkpoint.best_model_path).name)

In [ ]:
# 🖐️ 함께 따라하기: 새 LitCNN과 max_epochs=10인 Trainer(같은 CSVLogger 이름, gradient_clip_val=1.0)를 만들고, ckpt_path로 저장한 파일을 주어 이어 학습하세요
resumed_model = ...
resumed_trainer = ...
...
print("이어 학습 뒤 에폭 수:", resumed_trainer.current_epoch, "| 갱신 횟수:", resumed_trainer.global_step)

In [ ]:
# 끊기 전 기록(1~5에폭)과 이어 학습한 기록(6~10에폭)을 위아래로 붙입니다
resume_table = pd.concat([read_metrics(first_trainer), read_metrics(resumed_trainer)])
display(resume_table.set_index("epoch").round(4))

**결과 읽기**
- **`epoch=4-step=30.ckpt`:** 0부터 센 4번째, 곧 5번째 에폭이 끝난 때의 파일이고, 갱신 횟수 30은 6스텝 × 5에폭입니다(400장을 64장씩, 남는 16장은 버림).
- **에폭 10, 갱신 60번:** 새 Trainer는 1에폭이 아니라 6에폭부터 5에폭만 더 학습했고, 갱신 횟수도 30에 이어 60이 됐습니다.
- **`train_loss` 5에폭 1.1236, 6에폭 1.0915:** 6에폭 값이 1에폭(2.2357)으로 돌아가지 않고 5에폭에서 이어집니다. 파일에서 파라미터가 되살아났습니다(옵티마이저 상태와 에폭 수도 함께 되살아납니다).
- **끊지 않은 학습과의 차이:** 미니배치를 섞는 순서와 Dropout이 0으로 만들 뉴런을 정하는 난수 상태는 체크포인트에 없으므로, 끊지 않고 10에폭 학습한 값과 숫자까지 같지는 않습니다.

### 9-4. Dropout p 탐색

Dropout `p`를 0.25, 0.5, 0.75 세 후보로 학습해 최저 `val_loss`가 가장 낮은 `p`를 고릅니다(7-1의 격자 탐색). 후보마다 같은 시드, 같은 조기 종료(patience 10), 기울기 자르기(1.0)로 학습하고, 테스트 데이터는 쓰지 않습니다.

- **에폭 간 변화:** 2-3의 `epoch_change`는 `검증 손실` 열을 읽으므로, Lightning 기록의 `val_loss` 열 이름을 바꿔 넘깁니다.
- 세 번 학습하므로 CPU에서 30초 안팎 걸립니다.

In [ ]:
search_models = {}
search_trainers = {}
search_checkpoints = {}
search_rows = []
for dropout_p in [0.25, 0.5, 0.75]:
    L.seed_everything(0, verbose=False)

    # 🖐️ 함께 따라하기: 이 dropout_p로 LitCNN, 조기 종료(patience 10), val_loss 체크포인트, CSVLogger(name=f"lesson02_fashion_p{dropout_p}")를 만들고,
    # gradient_clip_val=1.0인 최대 100에폭 Trainer로 학습하세요
    p_model = ...
    p_early_stopping = ...
    p_checkpoint = ...
    p_trainer = ...
    ...

    # 후보마다 모델, Trainer, 체크포인트를 담아 두고, 기록에서 한 행을 만듭니다
    search_models[dropout_p] = p_model
    search_trainers[dropout_p] = p_trainer
    search_checkpoints[dropout_p] = p_checkpoint
    p_metrics = read_metrics(p_trainer)
    search_rows.append({"Dropout p": dropout_p, "멈춘 에폭": p_trainer.current_epoch,
                        "가장 좋았던 에폭": p_metrics.loc[p_metrics["val_loss"].idxmin(), "epoch"],
                        "최저 val_loss": p_checkpoint.best_model_score.item(),
                        "에폭 간 변화": epoch_change(p_metrics.rename(columns={"val_loss": "검증 손실"}))})

search_table = pd.DataFrame(search_rows)
display(search_table.round(4))

In [ ]:
# 🖐️ 함께 따라하기: 최저 val_loss가 가장 낮은 행의 Dropout p를 고르세요
chosen_p = ...
print("고른 Dropout p:", chosen_p)

In [ ]:
# 규제 없음과 세 후보의 val_loss 곡선을 겹쳐 그립니다
plt.figure(figsize=(8, 4.5))
plt.plot(fashion_plain_metrics["epoch"], fashion_plain_metrics["val_loss"], linewidth=2, label="규제 없음")
for dropout_p in search_trainers:
    p_metrics = read_metrics(search_trainers[dropout_p])
    plt.plot(p_metrics["epoch"], p_metrics["val_loss"], linewidth=2, label=f"Dropout {dropout_p}")
plt.ylim(0, 1.5)
plt.xlabel("에폭", fontsize=12)
plt.ylabel("val_loss", fontsize=12)
plt.title("옷 사진의 검증 손실(조기 종료, patience 10)", fontsize=14)
plt.legend(fontsize=11)
plt.show()

**결과 읽기**
- **최저 `val_loss`:** `p` 0.25가 23에폭 0.5167, 0.5가 45에폭 0.5208, 0.75가 46에폭 0.5659입니다. 규칙대로 가장 낮은 0.25를 골랐습니다.
- **차이와 에폭 간 변화:** 0.25와 0.5의 차이 0.004는 후보마다의 에폭 간 변화(0.0107~0.0210)보다 작아 설정의 차이로 단정할 수 없습니다. 0.75와의 차이 0.049는 그보다 큽니다.
- **`p`가 클수록 늦게 멈춤(이 탐색):** 0.25는 33에폭, 0.75는 56에폭에 멈췄습니다. 뉴런을 많이 0으로 만들수록 학습 그림을 천천히 외워 `val_loss`가 늦게까지 내려가기 쉽습니다.

### 9-5. 고른 모델의 검증 손실 확인과 테스트 한 번

고른 `p`의 가장 좋은 체크포인트를 파일에서 불러와, 평가 모드(`model.eval()`)와 기울기 기록 끄기(`torch.no_grad()`)로 검증 손실을 직접 계산해 체크포인트가 기록한 값과 비교합니다. `trainer.validate`와 `trainer.test`는 이 두 가지를 Lightning이 대신 합니다(8-1 표).

In [ ]:
# 고른 p의 가장 좋은 체크포인트 파일에서 모델을 새로 만들어 계산 장치에 불러옵니다
chosen_lit_model = LitCNN.load_from_checkpoint(search_checkpoints[chosen_p].best_model_path, map_location=device)

# 🖐️ 함께 따라하기: 모델을 평가 모드로 바꾸세요
...

# 🖐️ 기울기 기록을 끄는 with 블록 안에서 검증 그림 200장을 한 번에 넣어 로짓 valid_logits를 구하세요
...
chosen_valid_loss = loss_fn(valid_logits, y_fashion_valid.to(device)).item()
print("직접 계산한 val_loss:", round(chosen_valid_loss, 4), "| 체크포인트가 기록한 val_loss:", round(search_checkpoints[chosen_p].best_model_score.item(), 4))

`trainer.test(모델, dataloaders=테스트 DataLoader, ckpt_path="best", verbose=False)`는 8-6의 `trainer.validate`와 같은 방식으로 가장 좋은 체크포인트를 불러온 뒤 `test_step`을 돌려, `self.log`로 남긴 `test_loss`와 `test_acc`를 돌려줍니다. 고른 `p`의 Trainer로 한 번만 부릅니다.

In [ ]:
# 🖐️ 함께 따라하기: 고른 p의 Trainer와 모델로, 가장 좋은 체크포인트를 불러와 테스트 데이터를 한 번 평가하세요
fashion_test_results = ...
print("테스트 손실:", round(fashion_test_results[0]["test_loss"], 4), "| 테스트 정확도:", round(fashion_test_results[0]["test_acc"], 4))

**결과 읽기**
- **직접 계산한 `val_loss`:** 체크포인트가 기록한 값과 같습니다. `eval()`로 Dropout을 끄고 계산했기 때문이고, `no_grad()`는 값을 바꾸지 않고 역전파용 기록만 끕니다.
- **테스트 정확도 0.7915:** 고른 모델 하나만 한 번 평가한 값이고, 이 값이 보고할 성능입니다. 실행 환경에 따라 0.01 안팎 달라질 수 있습니다.

**확인 퀴즈**: 탐색에서 고른 `p` = 0.25가 다른 두 후보보다 좋은 설정이라고 말할 수 있나요? 세 후보를 모두 테스트 데이터로 평가해 확인하면 될까요?

<details><summary>정답 보기</summary>

- **좋다고 단정할 수 없습니다:** 0.5와의 차이(0.004)는 에폭 간 변화보다 작고, 0.75와의 차이(0.049)도 시드 하나로 한 번씩 학습한 결과입니다. 하나를 골라야 하므로 정한 규칙대로 가장 낮은 값을 골랐을 뿐입니다.
- **테스트로 확인하면 안 됩니다:** 테스트 결과로 후보를 비교하면 테스트 데이터가 고르는 데 쓴 데이터가 되어, 보고하는 테스트 성능이 실제보다 좋게 나옵니다(7-2 확인 퀴즈).
- **실무에서 확인하는 법:** 검증 데이터를 늘리거나, 시드를 바꿔 후보마다 여러 번 학습해 최저 `val_loss`의 평균과 흩어짐을 비교합니다.

</details>

## 10. 핵심 정리

- **학습·검증·테스트:** 학습 데이터로 파라미터를 갱신하고, 검증 데이터로 설정(규제, 멈출 에폭)을 고르고, 테스트 데이터는 고른 모델에 한 번만 씁니다(이 교안은 800, 200, 2,000장).
- **데이터 누수와 묶음 단위 분할:** 검증·테스트의 정보가 학습에 섞이면 성능이 실제보다 좋게 나옵니다. 같은 사람·같은 원본의 데이터는 묶음째 한쪽에만 넣고(과제 02는 손 모델 단위), 표준화 통계는 학습 데이터로만 계산합니다.
- **과적합 진단:** 파라미터 421,642개인 큰 CNN은 학습 그림 800장에서 학습 손실이 0.0024까지 줄었고, 검증 손실은 10에폭의 0.1685에서 30에폭의 0.2360으로 다시 커졌습니다. 학습 손실만 줄고 검증 손실이 다시 커지면 과적합이고, 둘 다 높으면 과소적합입니다.
- **Dropout:** 학습할 때 뉴런 출력을 확률 $p$로 0으로 만들고 남은 값을 $\frac{1}{1 - p}$배 합니다. 평가할 때는 끕니다. 그래서 `train_one_epoch`는 `model.train()`, `evaluate`는 `model.eval()`로 시작합니다. `model.eval()`은 층의 동작을 바꾸고 `torch.no_grad()`는 기울기 기록을 끄므로, 평가에는 둘 다 씁니다. $p$ = 0.5에서 가장 낮은 검증 손실이 0.1685에서 0.1179가 됐습니다.
- **Weight decay:** AdamW가 갱신마다 가중치를 학습률과 $\lambda$를 곱한 비율만큼 줄입니다. 가중치 크기는 규제 없음 약 14.6, $\lambda$ 0.01 약 14.5, 1.0 약 12.2, 10.0 약 8.0이었습니다. 기본값 0.01은 이 학습(학습률 0.001, 360번 갱신)에서 너무 약했고 10.0은 과소적합이 됐으므로 $\lambda$는 검증 손실로 고릅니다.
- **데이터 증강:** 정답이 바뀌지 않는 변환(숫자는 1~2픽셀 옮기기, 손 그림은 좌우 뒤집기)으로 에폭마다 학습 그림을 바꿉니다. 뒤집기, 회전, 이동, 자르기·크기 조절, 밝기·대비, 잡음 가운데 정답을 바꾸는 변환(숫자의 뒤집기, 6의 180° 회전)은 쓰지 않습니다. 30에폭 검증 손실이 0.0783으로 단독 규제 가운데 가장 낮았습니다.
- **조기 종료:** 검증 손실이 patience 에폭 동안 낮아지지 않으면 멈춥니다. 이 교안의 함수는 가장 낮을 때 파라미터를 `copy.deepcopy`로 저장해 두고 멈춘 뒤 되돌립니다(patience 10: 10에폭 저장, 20에폭 멈춤).
- **하이퍼파라미터 탐색:** 후보 조합을 같은 조건으로 학습해 검증 손실로 고르는 일입니다(격자 탐색, 무작위 탐색). 7-1은 Dropout `p`, λ, 증강의 후보 둘씩으로 만든 격자 8조합 가운데 다섯을 학습했습니다.
- **규제 비교와 테스트:** 다섯 설정을 같은 조기 종료로 비교하면 `증강`(0.0783)과 `Dropout + 증강`(0.0827)의 최저 검증 손실이 가장 낮습니다. 차이 0.0044가 에폭 간 변화(0.01~0.02)보다 작아 둘을 가를 수 없으므로 규칙대로 `증강`을 골랐고, 테스트 정확도는 0.9605입니다.
- **PyTorch Lightning:** `LightningModule`에 모델, `training_step`, `validation_step`, `configure_optimizers`를 쓰면 `Trainer`가 학습 반복과 모드 바꾸기를 하고, `EarlyStopping`, `ModelCheckpoint`, `CSVLogger`가 조기 종료, 저장, 기록을 맡습니다. `EarlyStopping`은 멈추기만 하므로, 가장 좋은 파라미터는 `ckpt_path="best"`로 불러옵니다(Dropout 0.5의 `val_loss` 0.1179).
- **기울기 자르기:** 노름 기준 자르기는 기울기 전체의 크기가 기준값보다 크면 같은 비율로 줄입니다((3, 4)는 기준값 1에서 (0.6, 0.8)). `loss.backward()`와 `optimizer.step()` 사이의 `clip_grad_norm_`, Lightning의 `gradient_clip_val`입니다.
- **옵티마이저 상태와 이어 학습:** Adam은 파라미터마다 기울기의 이동평균 $m$(`exp_avg`)과 기울기 제곱의 이동평균 $v$(`exp_avg_sq`)를 둡니다(상태의 숫자는 파라미터 수의 두 배). 체크포인트에 `optimizer_states`가 함께 있어 `trainer.fit(..., ckpt_path=파일 경로)`로 이어 학습합니다.
- **Hugging Face Trainer:** 같은 기능이 `eval_strategy`, `save_strategy`, `load_best_model_at_end`, `metric_for_best_model="eval_loss"`, `EarlyStoppingCallback`, `max_grad_norm`(기본 1.0)입니다(학습률, 에폭 수, weight decay는 딥러닝 학습 원리 단원 교안 02 7-4).